In [1]:
import os
import sys
import logging
import streamlit as st
import docx2txt
from crewai import Crew, Task, Agent, Process
from crewai_tools import ScrapeWebsiteTool
from jobfusion_agents import JobFusion_Agents
from jobfusion_tasks import JobFusion_Tasks
from jobfusion2_agents import JobFusion2_Agents
from jobfusion2_tasks import JobFusion2_Tasks
from langchain.chat_models import ChatOpenAI
from dotenv import load_dotenv
from Config import configure as cfg
from mock_interview_chatbot import *
import streamlit as st

import os
import requests
from crewai import Agent, Task, Crew, Process
from langchain.llms import OpenAI
import markdown
import PyPDF2
from fpdf import FPDF

from crewai_tools import MDXSearchTool, PDFSearchTool

/Users/frankwei/Documents/Side_Project/jobfusion_subj/jobfusion310/lib/python3.10/site-packages/langchain_core/_api/deprecation.py:119: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 0.2.0. An updated version of the class exists in the langchain-openai package and should be used instead. To use it run `pip install -U langchain-openai` and import as `from langchain_openai import ChatOpenAI`.
  warn_deprecated(


In [2]:
from dotenv import load_dotenv
load_dotenv()
openai_api_key = os.getenv('OPENAI_API_KEY')
llm_35_turbo = ChatOpenAI(api_key=openai_api_key, model='gpt-3.5-turbo', temperature=0.7, max_tokens=2000)
manager_llm_35_turbo = ChatOpenAI(api_key=openai_api_key, model='gpt-3.5-turbo')



## 1. Design an agent to extract information and then create the pdf file.

In [74]:
def create_resume_review_agent():

    Info_extract_agent = Agent(
        role='Resume information extracter',
        goal='Extract information from the provided resume files including summary, work experience, education, skills, name, address, email address, and phone number to a json file',
        backstory="""You are a skilled information extraction agent with expertise in parsing and analyzing resume data.""",
        verbose=True,
        allow_delegation=False,
        tools=[MDXSearchTool(mdx=resume_output_md_path)],
        llm=llm_35_turbo,
        )
    # code_agent = Agent(
    #             role='Coder',
    #             goal='Write the python code to combine all the information \
    #             from Info_extract_agent to the pdf format so that the sample resume',
    #             backstory="""You are a skilled coder with expertise in writing Python code to combine and format resume data.""",
    #             verbose=True,
    #             allow_delegation=False,
    #             tools=[PDFSearchTool(pdf=sample_pdf_path)],
    #             llm=OpenAI(temperature=0.7)
    #             )
    # code_agent = Agent(
    #             role='Coder',
    #             goal='Write the python code to combine all the information',
    #             backstory="""You are a skilled coder with expertise in writing Python code to combine and format resume data.""",
    #             verbose=True,
    #             allow_delegation=False,
    #             # tools=[PDFSearchTool(pdf=sample_pdf_path)],
    #             llm=llm_35_turbo,
    #             )
    Info_extract_task = Task(
                description= '''Forget everything you have learned. Extract information from the provided resume files including summary, work experience, education, skills, address,\
                      email address, and phone number to a json file. Make sure you are\
                          including all the working experiences. Name is always appearing in the first line of the resume without any prefix or name tag.\
                            When you are done with the work, take a deep breath and double check what you have done to make sure the output json file aligns with the schema including all the information.\
                            The schema you should be follow is: RESUME_SCHEMA = {
    "personal_info": {
        "name": str,
        "title": str,
        "summary": str,
        "contact": {
            "email": str,
            "phone": str,
            "location": str,
            "linkedin": str
        }
    },
    "work_experience": [{
        "company": str,
        "position": str,
        "duration": str,
        "location": str,
        "achievements": [str]
    }],
    "education": [{
        "school": str,
        "degree": str,
        "duration": str,
        "location": str
    }],
    "skills": [str]
}
When you are done with the work, go back to the md file again and check if you have missed anything. If you have missed anything, go back to the resume files and extract the missing information.\
    ''',
                agent=Info_extract_agent,
                expected_output="Json file with all the extracted information from the resume files",
                output_file='output/updated_resumev2.json',
                )
    # code_task = Task(
    #             description= "Write the python code to combine summary, work experience, education, skills, address, email address, and phone number to the pdf format similar to the sample resume",
    #             agent=code_agent,
    #             expected_output="Python code to combine and format resume data",
    #             output_file='output/code.txt',
    #             )
    resume_review_crew = Crew(
    agents=[Info_extract_agent],
    tasks=[Info_extract_task],
    process=Process.sequential,
    manager_llm = manager_llm_35_turbo, # Assign the manager_llm to the Crew
    verbose=True
    )
    return resume_review_crew

# if __name__ == "main":
resume_output_md_path = 'output/updated_resume.md'
# sample_pdf_path = 'output/amazon-data-science-resume-example.pdf'
create_resume_review_agent().kickoff()

2024-11-19 07:13:10,703 - 8482288192 - __init__.py-__init__:531 - WARNING: Overriding of current TracerProvider is not allowed


 [DEBUG]: == Working Agent: Resume information extracter
 [INFO]: == Starting Task: Forget everything you have learned. Extract information from the provided resume files including summary, work experience, education, skills, address,                      email address, and phone number to a json file. Make sure you are                          including all the working experiences. Name is always appearing in the first line of the resume without any prefix or name tag.                            When you are done with the work, take a deep breath and double check what you have done to make sure the output json file aligns with the schema including all the information.                            The schema you should be follow is: RESUME_SCHEMA = {
    "personal_info": {
        "name": str,
        "title": str,
        "summary": str,
        "contact": {
            "email": str,
            "phone": str,
            "location": str,
            "linkedin": str
        }
    },
    

'{\n    "personal_info": {\n        "name": "Diana Liu",\n        "title": "AI Data Scientist Freelancer",\n        "summary": "Proficient in mathematical statistics, econometrics, machine learning and deep learning. Offering 15 years of extensive project management expertise with a deep involvement in experimental design, data integration and cleansing, feature engineering, as well as mastery in machine learning, optimization, and deep learning implementations. With extensive experience in leveraging advanced AI technologies, I have successfully utilized prompt engineering, Retrieval-Augmented Generation (RAG), and agents to develop robust applications that address complex business challenges, automate processes, and drive business growth.",\n        "contact": {\n            "email": "lxjiao0805@gmail.com",\n            "phone": "202.739.1368",\n            "location": "Stanford, CA",\n            "linkedin": ""\n        }\n    },\n    "work_experience": [{\n            "company": "A

In [77]:
import json
from reportlab.lib import colors
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, ListItem, ListFlowable
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch

# Define standard resume format schema
RESUME_SCHEMA = {
    "personal_info": {
        "name": str,
        "title": str,
        "summary": str,
        "contact": {
            "email": str,
            "phone": str,
            "location": str,
            "linkedin": str
        }
    },
    "work_experience": [{
        "company": str,
        "position": str,
        "duration": str,
        "location": str,
        "achievements": [str]
    }],
    "education": [{
        "school": str,
        "degree": str,
        "duration": str,
        "location": str
    }],
    "skills": [str]
}

def validate_resume_format(data):
    """Validate if the input data matches the schema"""
    try:
        for key, value_type in RESUME_SCHEMA.items():
            if key not in data:
                raise ValueError(f"Missing required section: {key}")
            
            if isinstance(value_type, list):
                if not isinstance(data[key], list):
                    raise ValueError(f"{key} must be a list")
                
                for item in data[key]:
                    for subkey, subtype in value_type[0].items():
                        if subkey not in item:
                            raise ValueError(f"Missing required field {subkey} in {key}")
                        if not isinstance(item[subkey], subtype):
                            raise ValueError(f"Invalid type for {subkey} in {key}")
            
            elif isinstance(value_type, dict):
                for subkey, subtype in value_type.items():
                    if subkey not in data[key]:
                        raise ValueError(f"Missing required field {subkey} in {key}")
                    if not isinstance(data[key][subkey], subtype):
                        raise ValueError(f"Invalid type for {subkey} in {key}")
        
        return True
    except Exception as e:
        print(f"Validation error: {str(e)}")
        return False

def parse_resume_to_json(markdown_text):
    """Parse markdown text to JSON format"""
    # Implementation would depend on the specific markdown format
    # This is a placeholder for the parsing logic
    resume_data = {}  # Parse markdown_text to fill this dictionary
    return resume_data

def create_pdf_from_json(json_data, output_filename):
    """Convert JSON data to PDF format"""
    doc = SimpleDocTemplate(output_filename, pagesize=letter,
                          rightMargin=72, leftMargin=72,
                          topMargin=72, bottomMargin=18)
    
    styles = getSampleStyleSheet()
    story = []
    
    # Custom styles
    styles.add(ParagraphStyle(name='Name',
                            fontSize=24,
                            spaceAfter=30))
    styles.add(ParagraphStyle(name='Section',
                            fontSize=16,
                            spaceBefore=20,
                            spaceAfter=12))
    
    # Generate PDF sections based on json_data
    # Personal Info
    story.append(Paragraph(json_data["personal_info"]["name"], styles["Name"]))
    story.append(Paragraph(json_data["personal_info"]["title"], styles["Heading2"]))
    story.append(Paragraph(json_data["personal_info"]["summary"], styles["Normal"]))
    
    # Contact Info
    contact = json_data["personal_info"]["contact"]
    contact_text = f"{contact['email']} | {contact['phone']} | {contact['location']} | {contact['linkedin']}"
    story.append(Paragraph(contact_text, styles["Normal"]))
    story.append(Spacer(1, 20))
    
    # Work Experience
    story.append(Paragraph("Work Experience", styles["Section"]))
    for job in json_data["work_experience"]:
        story.append(Paragraph(f"<b>{job['position']}</b> - {job['company']}", styles["Normal"]))
        story.append(Paragraph(f"{job['duration']} | {job['location']}", styles["Normal"]))
        achievements = [ListItem(Paragraph(item, styles["Normal"])) for item in job["achievements"]]
        story.append(ListFlowable(achievements, bulletType='bullet'))
        story.append(Spacer(1, 12))
    
    # Education
    story.append(Paragraph("Education", styles["Section"]))
    for edu in json_data["education"]:
        story.append(Paragraph(f"<b>{edu['school']}</b> - {edu['degree']}", styles["Normal"]))
        story.append(Paragraph(f"{edu['duration']} | {edu['location']}", styles["Normal"]))
    
    # Skills
    story.append(Paragraph("Skills", styles["Section"]))
    skills_text = "; ".join(json_data["skills"])
    story.append(Paragraph(skills_text, styles["Normal"]))
    
    doc.build(story)

def convert_resume(markdown_text, json_output="resume.json", pdf_output="resume.pdf"):
    """Main function to handle the conversion process"""
    # Convert to JSON
    # resume_json = parse_resume_to_json(markdown_text)
    resume_json = markdown_text
    # Validate format
    if not validate_resume_format(resume_json):
        raise ValueError("Invalid resume format")
    
    # Save JSON file
    with open(json_output, 'w') as f:
        json.dump(resume_json, f, indent=2)
    
    # Create PDF
    create_pdf_from_json(resume_json, pdf_output)
    
    return resume_json

if __name__=="__main__":
    resume_output_md_path = 'output/updated_resume.md'
    resume_input_json_path = 'output/updated_resumev2.json'
    # Load the markdown resume text
    with open(resume_output_md_path, 'r') as f:
        resume_text = f.read()

    with open(resume_input_json_path, 'r') as f:
        resume_json = f.read()
    
    # Convert the resume text to JSON and PDF
    convert_resume(resume_json, json_output="output/resume_v2.json", pdf_output="output/resume_v2.pdf")
    st.write("Resume conversion completed successfully!")
    st.write("You can download the JSON and PDF files below:")
    st.markdown("[Download JSON](output/resume.json)")
    st.markdown("[Download PDF](output/resume.pdf)")

Validation error: string indices must be integers


ValueError: Invalid resume format

In [81]:
def create_pdf_from_json(json_data, output_filename):
    if isinstance(json_data, str):
        json_data = json.loads(json_data)
    
    doc = SimpleDocTemplate(output_filename, pagesize=letter,
                          rightMargin=72, leftMargin=72,
                          topMargin=72, bottomMargin=18)
    
    styles = getSampleStyleSheet()
    story = []
    
    styles.add(ParagraphStyle(name='Name',
                            fontSize=24,
                            spaceAfter=10))  # Reduced space after name
    styles.add(ParagraphStyle(name='Contact',
                            fontSize=12,
                            spaceAfter=20))  # Added contact style
    styles.add(ParagraphStyle(name='Section',
                            fontSize=16,
                            spaceBefore=20,
                            spaceAfter=12))
    
    # Personal Info with contact first
    if isinstance(json_data.get("personal_info"), dict):
        person = json_data["personal_info"]
        
        # Name and contact info at top
        story.append(Paragraph(person.get("name", "No Name Provided"), styles["Name"]))
        
        # Contact Info before title
        if isinstance(person.get("contact"), dict):
            contact = person["contact"]
            contact_items = []
            if contact.get('email'): contact_items.append(contact['email'])
            if contact.get('phone'): contact_items.append(contact['phone'])
            if contact.get('location'): contact_items.append(contact['location'])
            if contact.get('linkedin'): contact_items.append(contact['linkedin'])
            
            contact_text = " | ".join(contact_items) if contact_items else "No Contact Information Provided"
            story.append(Paragraph(contact_text, styles["Contact"]))
        
        # Title and summary after contact
        story.append(Paragraph(person.get("title", "No Title Provided"), styles["Heading2"]))
        story.append(Paragraph(person.get("summary", "No Summary Provided"), styles["Normal"]))
    
    story.append(Spacer(1, 20))
    
    # Rest of the sections remain the same
    # Work Experience
    story.append(Paragraph("Work Experience", styles["Section"]))
    if isinstance(json_data.get("work_experience"), list) and json_data["work_experience"]:
        for job in json_data["work_experience"]:
            job_title = []
            if job.get('position'): job_title.append(f"<b>{job['position']}</b>")
            if job.get('company'): job_title.append(job['company'])
            job_title_text = " - ".join(job_title) if job_title else "Position Details Not Available"
            story.append(Paragraph(job_title_text, styles["Normal"]))
            
            location_info = []
            if job.get('duration'): location_info.append(job['duration'])
            if job.get('location'): location_info.append(job['location'])
            location_text = " | ".join(location_info) if location_info else "Location/Duration Not Available"
            story.append(Paragraph(location_text, styles["Normal"]))
            
            if isinstance(job.get("achievements"), list) and job["achievements"]:
                achievements = [ListItem(Paragraph(item, styles["Normal"])) 
                              for item in job["achievements"] if item]
                if achievements:
                    story.append(ListFlowable(achievements, bulletType='bullet'))
            story.append(Spacer(1, 12))
    else:
        story.append(Paragraph("No work experience provided", styles["Normal"]))
    
    # Education
    story.append(Paragraph("Education", styles["Section"]))
    if isinstance(json_data.get("education"), list) and json_data["education"]:
        for edu in json_data["education"]:
            edu_info = []
            if edu.get('school'): edu_info.append(f"<b>{edu['school']}</b>")
            if edu.get('degree'): edu_info.append(edu['degree'])
            edu_text = " - ".join(edu_info) if edu_info else "Education Details Not Available"
            story.append(Paragraph(edu_text, styles["Normal"]))
            
            edu_details = []
            if edu.get('duration'): edu_details.append(edu['duration'])
            if edu.get('location'): edu_details.append(edu['location'])
            details_text = " | ".join(edu_details) if edu_details else "Duration/Location Not Available"
            story.append(Paragraph(details_text, styles["Normal"]))
    else:
        story.append(Paragraph("No education information provided", styles["Normal"]))
    
    # Skills
    story.append(Paragraph("Skills", styles["Section"]))
    if isinstance(json_data.get("skills"), list) and json_data["skills"]:
        skills = [skill for skill in json_data["skills"] if skill]
        skills_text = "; ".join(skills) if skills else "No specific skills listed"
        story.append(Paragraph(skills_text, styles["Normal"]))
    else:
        story.append(Paragraph("No skills provided", styles["Normal"]))
    
    doc.build(story)
create_pdf_from_json(resume_json, output_filename="output/resume_v2.pdf")

In [83]:
def create_pdf_from_json(json_data, output_filename):
    if isinstance(json_data, str):
        json_data = json.loads(json_data)
    
    doc = SimpleDocTemplate(output_filename, pagesize=letter,
                          rightMargin=72, leftMargin=72,
                          topMargin=72, bottomMargin=18)
    
    styles = getSampleStyleSheet()
    story = []
    
    styles.add(ParagraphStyle(name='Name',
                            fontSize=24,
                            spaceAfter=12))  # Increased space after name
    styles.add(ParagraphStyle(name='Contact',
                            fontSize=12,
                            spaceAfter=20))
    styles.add(ParagraphStyle(name='Section',
                            fontSize=16,
                            spaceBefore=20,
                            spaceAfter=12))
    
    # Personal Info
    if isinstance(json_data.get("personal_info"), dict):
        person = json_data["personal_info"]
        story.append(Paragraph(person.get("name", "No Name Provided"), styles["Name"]))
        
        if isinstance(person.get("contact"), dict):
            contact = person["contact"]
            contact_items = []
            if contact.get('email'): contact_items.append(contact['email'])
            if contact.get('phone'): contact_items.append(contact['phone'])
            if contact.get('location'): contact_items.append(contact['location'])
            if contact.get('linkedin'): contact_items.append(contact['linkedin'])
            
            contact_text = " | ".join(contact_items) if contact_items else "No Contact Information Provided"
            story.append(Paragraph(contact_text, styles["Contact"]))
        
        # story.append(Paragraph(person.get("title", "No Title Provided"), styles["Heading2"]))
        story.append(Paragraph('Summary', styles["Heading2"]))
        story.append(Paragraph(person.get("summary", "No Summary Provided"), styles["Normal"]))
    
    story.append(Spacer(1, 20))
    
    # Work Experience
    story.append(Paragraph("Work Experience", styles["Section"]))
    if isinstance(json_data.get("work_experience"), list) and json_data["work_experience"]:
        for job in json_data["work_experience"]:
            job_title = []
            if job.get('position'): job_title.append(f"<b>{job['position']}</b>")
            if job.get('company'): job_title.append(job['company'])
            job_title_text = " - ".join(job_title) if job_title else "Position Details Not Available"
            story.append(Paragraph(job_title_text, styles["Normal"]))
            
            location_info = []
            if job.get('duration'): location_info.append(job['duration'])
            if job.get('location'): location_info.append(job['location'])
            location_text = " | ".join(location_info) if location_info else "Location/Duration Not Available"
            story.append(Paragraph(location_text, styles["Normal"]))
            
            if isinstance(job.get("achievements"), list) and job["achievements"]:
                achievements = [ListItem(Paragraph(item, styles["Normal"])) 
                              for item in job["achievements"] if item]
                if achievements:
                    story.append(ListFlowable(achievements, bulletType='bullet'))
            story.append(Spacer(1, 12))
    else:
        story.append(Paragraph("No work experience provided", styles["Normal"]))
    
    # Education with reordered fields
    story.append(Paragraph("Education", styles["Section"]))
    if isinstance(json_data.get("education"), list) and json_data["education"]:
        for edu in json_data["education"]:
            # First line: Degree
            if edu.get('degree'):
                story.append(Paragraph(f"<b>{edu['degree']}</b>", styles["Normal"]))
            
            # Second line: Duration, School, Location
            edu_details = []
            if edu.get('duration'): edu_details.append(edu['duration'])
            if edu.get('school'): edu_details.append(edu['school'])
            if edu.get('location'): edu_details.append(edu['location'])
            
            details_text = " | ".join(edu_details) if edu_details else "Education Details Not Available"
            story.append(Paragraph(details_text, styles["Normal"]))
            story.append(Spacer(1, 8))
    else:
        story.append(Paragraph("No education information provided", styles["Normal"]))
    
    # Skills
    story.append(Paragraph("Skills", styles["Section"]))
    if isinstance(json_data.get("skills"), list) and json_data["skills"]:
        skills = [skill for skill in json_data["skills"] if skill]
        skills_text = "; ".join(skills) if skills else "No specific skills listed"
        story.append(Paragraph(skills_text, styles["Normal"]))
    else:
        story.append(Paragraph("No skills provided", styles["Normal"]))
    
    doc.build(story)

create_pdf_from_json(resume_json, output_filename="output/resume_v4.pdf")

In [60]:
import json
from typing import Dict, List, Any
import re

class ResumeParser:
    def __init__(self):
        # 基础关键词定义
        self.name_keywords = ['name', 'full_name', 'full name', 'candidate_name', 'candidate name']
        self.contact_keywords = ['contact', 'contact_info', 'contact info', 'contact_information']
        self.email_keywords = ['email', 'e-mail', 'email_address']
        self.phone_keywords = ['phone', 'telephone', 'mobile', 'phone_number', 'contact_number']
        self.location_keywords = ['location', 'address', 'city', 'residence']
        self.experience_keywords = ['experience', 'work_experience', 'employment', 'work history', 'jobs']
        self.education_keywords = ['education', 'academic', 'qualification', 'study']
        self.skills_keywords = ['skills', 'technical skills', 'competencies', 'expertise']
        
        # 教育相关关键词
        self.school_keywords = ['school', 'university', 'institution', 'college', 'academy']
        self.degree_keywords = ['degree', 'qualification', 'major', 'field', 'concentration', 'program']
        self.date_keywords = ['date', 'period', 'year', 'duration', 'time']
        
        # 工作描述相关关键词
        self.description_keywords = [
            'description', 'responsibilities', 'duties', 'achievements', 
            'accomplishments', 'highlights', 'bullet_points', 'details',
            'tasks', 'projects', 'contributions'
        ]

    def find_key_by_patterns(self, data: dict, patterns: List[str]) -> str:
        """在字典中查找匹配特定模式的键"""
        if not isinstance(data, dict):
            return None
            
        for key in data.keys():
            key_lower = key.lower()
            for pattern in patterns:
                if pattern.lower() in key_lower:
                    return key
        return None

    def extract_contact_info(self, data: Dict) -> Dict:
        """提取联系信息"""
        contact_info = {}
        
        # 直接从数据中查找
        if isinstance(data, dict):
            for key, value in data.items():
                key_lower = key.lower()
                if any(keyword in key_lower for keyword in self.email_keywords):
                    contact_info['email'] = str(value)
                elif any(keyword in key_lower for keyword in self.phone_keywords):
                    contact_info['phone'] = str(value)
                elif any(keyword in key_lower for keyword in self.location_keywords):
                    contact_info['location'] = str(value)
        
        # 检查嵌套的联系信息
        contact_key = self.find_key_by_patterns(data, self.contact_keywords)
        if contact_key and isinstance(data[contact_key], dict):
            nested_contact = data[contact_key]
            for key, value in nested_contact.items():
                key_lower = key.lower()
                if any(keyword in key_lower for keyword in self.email_keywords):
                    contact_info['email'] = str(value)
                elif any(keyword in key_lower for keyword in self.phone_keywords):
                    contact_info['phone'] = str(value)
                elif any(keyword in key_lower for keyword in self.location_keywords):
                    contact_info['location'] = str(value)
        
        # 确保所有必需字段都有值
        for field in ['email', 'phone', 'location']:
            if field not in contact_info:
                contact_info[field] = ''
                
        return contact_info

    def extract_bullet_points(self, text: Any) -> List[str]:
        """从文本中提取要点"""
        bullet_points = []
        
        if isinstance(text, str):
            # 处理单个字符串
            separators = ['\n', ';', '。', '.']
            points = [text]
            for sep in separators:
                new_points = []
                for point in points:
                    new_points.extend(p.strip() for p in point.split(sep) if p.strip())
                points = new_points
            bullet_points.extend(points)
            
        elif isinstance(text, list):
            # 处理列表
            for item in text:
                if isinstance(item, str):
                    bullet_points.append(item.strip())
                elif isinstance(item, dict):
                    for value in item.values():
                        if isinstance(value, str):
                            bullet_points.append(value.strip())
                        elif isinstance(value, list):
                            bullet_points.extend(str(v).strip() for v in value if v)
        
        elif isinstance(text, dict):
            # 处理字典
            for value in text.values():
                if isinstance(value, str):
                    bullet_points.append(value.strip())
                elif isinstance(value, list):
                    bullet_points.extend(str(v).strip() for v in value if v)
        
        # 清理和标准化bullet points
        bullet_points = [
            point for point in bullet_points 
            if point and len(point) > 5 
            and not point.isspace()
        ]
        
        # 确保每个point是完整的句子
        bullet_points = [
            self.standardize_bullet_point(point)
            for point in bullet_points
        ]
        
        return bullet_points

    def standardize_bullet_point(self, bullet: str) -> str:
        """标准化bullet point的格式"""
        # 去除开头的特殊字符
        bullet = re.sub(r'^[-•*]+\s*', '', bullet.strip())
        
        # 确保第一个字母大写
        if bullet:
            bullet = bullet[0].upper() + bullet[1:]
            
        # 确保结尾有标点符号
        if bullet and not bullet[-1] in '.!?':
            bullet += '.'
            
        # 移除多余的空格
        bullet = ' '.join(bullet.split())
        
        return bullet

    def _process_experience_entry(self, exp: Dict) -> Dict:
        """处理单个工作经验条目"""
        experience = {}
        
        # 提取基本信息
        for key_list, field in [
            (self.name_keywords, 'company'),
            (['title', 'position', 'role'], 'title'),
            (self.date_keywords, 'date'),
            (self.location_keywords, 'location')
        ]:
            for key in key_list:
                if key in exp:
                    experience[field] = str(exp[key])
                    break
        
        # 提取工作描述
        bullets = []
        
        # 从各种可能的来源提取描述
        for key in self.description_keywords:
            if key in exp:
                extracted_bullets = self.extract_bullet_points(exp[key])
                if extracted_bullets:
                    bullets.extend(extracted_bullets)
        
        # 检查嵌套的描述
        for key, value in exp.items():
            if isinstance(value, dict):
                for desc_key in self.description_keywords:
                    if desc_key in value:
                        extracted_bullets = self.extract_bullet_points(value[desc_key])
                        if extracted_bullets:
                            bullets.extend(extracted_bullets)
            elif isinstance(value, list) and key not in ['company', 'title', 'date', 'location']:
                extracted_bullets = self.extract_bullet_points(value)
                if extracted_bullets:
                    bullets.extend(extracted_bullets)
        
        # 去重并清理
        bullets = list(dict.fromkeys(bullets))
        bullets = [b for b in bullets if len(b.split()) > 3]
        
        experience['achievements'] = bullets
        
        # 确保所有必需字段都有值
        for field in ['company', 'title', 'date', 'location', 'achievements']:
            if field not in experience:
                experience[field] = '' if field != 'achievements' else []
        
        return experience

    def _process_education(self, edu_data: Any) -> List[Dict]:
        """处理教育信息"""
        education = []
        
        if isinstance(edu_data, list):
            for edu in edu_data:
                if isinstance(edu, dict):
                    education_entry = self._process_education_entry(edu)
                    if education_entry:
                        education.append(education_entry)
        elif isinstance(edu_data, dict):
            if any(key in edu_data for key in self.school_keywords + self.degree_keywords):
                education_entry = self._process_education_entry(edu_data)
                if education_entry:
                    education.append(education_entry)
            else:
                for key, value in edu_data.items():
                    if isinstance(value, dict):
                        education_entry = self._process_education_entry(value)
                        if education_entry:
                            education.append(education_entry)
        
        return education

    def _process_education_entry(self, edu: Dict) -> Dict:
        """处理单个教育经历条目"""
        education_entry = {}
        
        # 提取各个字段
        for key_list, field in [
            (self.school_keywords, 'school'),
            (self.degree_keywords, 'degree'),
            (self.date_keywords, 'date'),
            (self.location_keywords, 'location')
        ]:
            for key in key_list:
                if key in edu:
                    value = edu[key]
                    if isinstance(value, dict):
                        # 处理嵌套结构
                        if field == 'degree':
                            parts = []
                            for d_key in ['level', 'major', 'field']:
                                if d_key in value:
                                    parts.append(str(value[d_key]))
                            education_entry[field] = ' '.join(parts)
                        elif field == 'date':
                            start = value.get('start', '')
                            end = value.get('end', '')
                            education_entry[field] = f"{start} - {end}" if start and end else end or start
                        elif field == 'location':
                            parts = []
                            for l_key in ['city', 'state', 'country']:
                                if l_key in value:
                                    parts.append(str(value[l_key]))
                            education_entry[field] = ', '.join(parts)
                    else:
                        education_entry[field] = str(value)
                    break
        
        # 确保所有必需字段都有值
        for field in ['school', 'degree', 'date', 'location']:
            if field not in education_entry:
                education_entry[field] = ''
        
        return education_entry

    def normalize_json_format(self, input_json: Dict) -> Dict:
        """将任意格式的JSON转换为标准格式"""
        normalized = {
            "name": "",
            "title": "",
            "contact": {},
            "experience": [],
            "education": [],
            "skills": []
        }
        
        # 提取姓名
        name_key = self.find_key_by_patterns(input_json, self.name_keywords)
        if name_key:
            normalized["name"] = str(input_json[name_key])
        
        # 提取职位
        title_key = self.find_key_by_patterns(input_json, ['title', 'position', 'role'])
        if title_key:
            normalized["title"] = str(input_json[title_key])
        
        # 提取联系信息
        normalized["contact"] = self.extract_contact_info(input_json)
        
        # 提取工作经验
        exp_key = self.find_key_by_patterns(input_json, self.experience_keywords)
        if exp_key:
            exp_data = input_json[exp_key]
            if isinstance(exp_data, list):
                for exp in exp_data:
                    if isinstance(exp, dict):
                        processed_exp = self._process_experience_entry(exp)
                        if processed_exp:
                            normalized["experience"].append(processed_exp)
            elif isinstance(exp_data, dict):
                processed_exp = self._process_experience_entry(exp_data)
                if processed_exp:
                    normalized["experience"].append(processed_exp)
        
        # 提取教育信息
        edu_key = self.find_key_by_patterns(input_json, self.education_keywords)
        if edu_key:
            normalized["education"] = self._process_education(input_json[edu_key])
        
        # 提取技能
        skills_key = self.find_key_by_patterns(input_json, self.skills_keywords)
        if skills_key:
            skills = input_json[skills_key]
            if isinstance(skills, str):
                normalized["skills"] = [skill.strip() for skill in skills.split(',')]
            elif isinstance(skills, list):
                normalized["skills"] = [str(skill).strip() for skill in skills]
            elif isinstance(skills, dict):
                skill_list = []
                for value in skills.values():
                    if isinstance(value, str):
                        skill_list.extend(value.split(','))
                    elif isinstance(value, list):
                        skill_list.extend(value)
                normalized["skills"] = [str(skill).strip() for skill in skill_list]
        
        return normalized

def process_resume(input_file: str, output_file: str) -> Dict:
    """处理简历文件"""
    try:
        # 读取输入文件
        with open(input_file, 'r', encoding='utf-8') as f:
            input_data = json.load(f)
        
        # 创建解析器并处理数据
        parser = ResumeParser()
        normalized_data = parser.normalize_json_format(input_data)
        
        # 保存处理后的数据
        with open(output_file, 'w', encoding='utf-8') as f:
            json.dump(normalized_data, f, indent=2, ensure_ascii=False)
        
        return normalized_data
        
    except Exception as e:
        print(f"Error processing resume: {str(e)}")
        raise

# 使用示例
    # 示例数据

if __name__ == "__main__":
    # 示例输入
    with open('output/updated_resume.json', 'r') as file:
        sample_input = json.load(file)
    
    # 解析和转换
    # parser = ResumeParser()
    # normalized = parser.normalize_json_format(sample_input)
    parser = ResumeParser()
    result = parser.normalize_json_format(sample_input)
    # 打印结果
    print(json.dumps(result, indent=2))

{
  "name": "Diana Liu",
  "title": "AI Data Scientist Freelancer",
  "contact": {
    "email": "lxjiao0805@gmail.com",
    "phone": "202.739.1368",
    "location": ""
  },
  "experience": [
    {
      "title": "AI Data Scientist Freelancer",
      "date": "March 2023 \u2013 present",
      "location": "Remote",
      "achievements": [
        "Enhance product descriptions and recommendations through various recommender systems.",
        "Support development and evaluation of internal survey automation pipeline which improves the efficiency of data extract, transform, load (ETL), anomalous detection, weighting and exporting.",
        "Collaborate with business development consultants to develop innovative winning solutions leveraging data science, conduct technical innovation to ensure statistical rigor in econometrics and modeling projects, and give internal team technical training on analytics topics like recommender system and imbalance data modeling.",
        "Build a query gen

In [ ]:
import json
from reportlab.lib import colors
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, ListItem, ListFlowable
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch

# Define standard resume format schema
RESUME_SCHEMA = {
    "personal_info": {
        "name": str,
        "title": str,
        "summary": str,
        "contact": {
            "email": str,
            "phone": str,
            "location": str,
            "linkedin": str
        }
    },
    "work_experience": [{
        "company": str,
        "position": str,
        "duration": str,
        "location": str,
        "achievements": [str]
    }],
    "education": [{
        "school": str,
        "degree": str,
        "duration": str,
        "location": str
    }],
    "skills": [str]
}

def validate_resume_format(data):
    """Validate if the input data matches the schema"""
    try:
        for key, value_type in RESUME_SCHEMA.items():
            if key not in data:
                raise ValueError(f"Missing required section: {key}")
            
            if isinstance(value_type, list):
                if not isinstance(data[key], list):
                    raise ValueError(f"{key} must be a list")
                
                for item in data[key]:
                    for subkey, subtype in value_type[0].items():
                        if subkey not in item:
                            raise ValueError(f"Missing required field {subkey} in {key}")
                        if not isinstance(item[subkey], subtype):
                            raise ValueError(f"Invalid type for {subkey} in {key}")
            
            elif isinstance(value_type, dict):
                for subkey, subtype in value_type.items():
                    if subkey not in data[key]:
                        raise ValueError(f"Missing required field {subkey} in {key}")
                    if not isinstance(data[key][subkey], subtype):
                        raise ValueError(f"Invalid type for {subkey} in {key}")
        
        return True
    except Exception as e:
        print(f"Validation error: {str(e)}")
        return False

def parse_resume_to_json(markdown_text):
    """Parse markdown text to JSON format"""
    # Implementation would depend on the specific markdown format
    # This is a placeholder for the parsing logic
    resume_data = {}  # Parse markdown_text to fill this dictionary
    return resume_data

def create_pdf_from_json(json_data, output_filename):
    """Convert JSON data to PDF format"""
    doc = SimpleDocTemplate(output_filename, pagesize=letter,
                          rightMargin=72, leftMargin=72,
                          topMargin=72, bottomMargin=18)
    
    styles = getSampleStyleSheet()
    story = []
    
    # Custom styles
    styles.add(ParagraphStyle(name='Name',
                            fontSize=24,
                            spaceAfter=30))
    styles.add(ParagraphStyle(name='Section',
                            fontSize=16,
                            spaceBefore=20,
                            spaceAfter=12))
    
    # Generate PDF sections based on json_data
    # Personal Info
    story.append(Paragraph(json_data["personal_info"]["name"], styles["Name"]))
    story.append(Paragraph(json_data["personal_info"]["title"], styles["Heading2"]))
    story.append(Paragraph(json_data["personal_info"]["summary"], styles["Normal"]))
    
    # Contact Info
    contact = json_data["personal_info"]["contact"]
    contact_text = f"{contact['email']} | {contact['phone']} | {contact['location']} | {contact['linkedin']}"
    story.append(Paragraph(contact_text, styles["Normal"]))
    story.append(Spacer(1, 20))
    
    # Work Experience
    story.append(Paragraph("Work Experience", styles["Section"]))
    for job in json_data["work_experience"]:
        story.append(Paragraph(f"<b>{job['position']}</b> - {job['company']}", styles["Normal"]))
        story.append(Paragraph(f"{job['duration']} | {job['location']}", styles["Normal"]))
        achievements = [ListItem(Paragraph(item, styles["Normal"])) for item in job["achievements"]]
        story.append(ListFlowable(achievements, bulletType='bullet'))
        story.append(Spacer(1, 12))
    
    # Education
    story.append(Paragraph("Education", styles["Section"]))
    for edu in json_data["education"]:
        story.append(Paragraph(f"<b>{edu['school']}</b> - {edu['degree']}", styles["Normal"]))
        story.append(Paragraph(f"{edu['duration']} | {edu['location']}", styles["Normal"]))
    
    # Skills
    story.append(Paragraph("Skills", styles["Section"]))
    skills_text = "; ".join(json_data["skills"])
    story.append(Paragraph(skills_text, styles["Normal"]))
    
    doc.build(story)

def convert_resume(markdown_text, json_output="resume.json", pdf_output="resume.pdf"):
    """Main function to handle the conversion process"""
    # Convert to JSON
    resume_json = parse_resume_to_json(markdown_text)
    
    # Validate format
    if not validate_resume_format(resume_json):
        raise ValueError("Invalid resume format")
    
    # Save JSON file
    with open(json_output, 'w') as f:
        json.dump(resume_json, f, indent=2)
    
    # Create PDF
    create_pdf_from_json(resume_json, pdf_output)
    
    return resume_json

In [52]:
from fpdf import FPDF
import json
from typing import Dict, List, Any

class GenericPDFGenerator:
    def __init__(self, page_width: int = 210, page_height: int = 297):
        """Initialize the PDF generator with A4 size by default"""
        self.pdf = FPDF(format=(page_width, page_height))
        self.pdf.add_page()
        self.pdf.set_auto_page_break(auto=True, margin=15)
        self.pdf.set_font('Helvetica')
        self.left_margin = 20
        self.current_y = 20

    def format_key(self, key: str) -> str:
        """Format dictionary keys into readable titles"""
        return ' '.join(word.capitalize() for word in key.split('_')).strip()

    def clean_text(self, text: str) -> str:
        """Clean text by replacing problematic characters with ASCII equivalents"""
        chars_to_replace = {
            '"': '"',
            '"': '"',
            ''': "'",
            ''': "'",
            '–': '-',
            '—': '-',
            '•': '*',
            '…': '...',
            '\u2013': '-',
            '\u2014': '-',
            '\u2018': "'",
            '\u2019': "'",
            '\u201C': '"',
            '\u201D': '"',
            '\u2026': '...',
            '\u2022': '*',
            '\xa0': ' ',
        }
        
        for old, new in chars_to_replace.items():
            text = text.replace(old, new)
            
        # Replace any remaining non-ASCII characters with their closest ASCII equivalent
        return text.encode('ascii', 'replace').decode('ascii')

    def add_text(self, text: str, font_size: int = 10, spacing: int = 5, style: str = '') -> None:
        """Add text with specified font size and spacing"""
        self.pdf.set_font('Helvetica', style, font_size)
        self.pdf.set_xy(self.left_margin, self.current_y)
        clean_text = self.clean_text(str(text))
        self.pdf.multi_cell(0, spacing, clean_text)
        self.current_y = self.pdf.get_y() + spacing

    def format_value(self, value: Any, indent_level: int = 0) -> None:
        """
        Recursively format any value based on its type
        """
        indent = self.left_margin + (indent_level * 10)
        max_width = 190 - indent  # Maximum width for text, accounting for page margins
        
        if isinstance(value, dict):
            for k, v in value.items():
                if not k.startswith('_'):  # Skip private fields
                    self.pdf.set_xy(indent, self.current_y)
                    if isinstance(v, (dict, list)):
                        self.add_text(self.format_key(k) + ':', 12, style='B')
                        self.format_value(v, indent_level + 1)
                    else:
                        combined_text = f"{self.format_key(k)}: {v}"
                        self.pdf.set_xy(indent, self.current_y)
                        self.add_text(combined_text, 10)

        elif isinstance(value, list):
            for item in value:
                if isinstance(item, dict):
                    self.format_value(item, indent_level)
                    self.current_y += 5
                else:
                    self.pdf.set_xy(indent, self.current_y)
                    self.pdf.cell(5, 5, "*", ln=0)
                    self.pdf.set_xy(indent + 5, self.current_y)
                    self.add_text(str(item), 10)

        else:
            self.pdf.set_xy(indent, self.current_y)
            self.add_text(str(value), 10)

    def create_section(self, key: str, value: Any) -> None:
        """Create a section for each top-level key in the JSON"""
        # Special handling for name and title if they exist
        if key.lower() == 'name':
            self.add_text(value, 24, style='B')
            return
        elif key.lower() == 'title':
            self.add_text(value, 16)
            return

        # Add section header with spacing
        self.current_y += 5
        self.add_text(self.format_key(key), 14, style='B')
        self.current_y += 2

        # Format section content
        self.format_value(value)
        self.current_y += 5

def json_to_pdf(json_data: str, output_file: str = "output.pdf") -> None:
    """
    Convert any JSON data to PDF format
    
    Args:
        json_data (str): JSON string containing data
        output_file (str): Output PDF file path
    """
    try:
        # Parse JSON data
        data = json.loads(json_data)
        
        # Initialize PDF generator
        generator = GenericPDFGenerator()
        
        # Process each top-level key in the JSON
        for key, value in data.items():
            if not key.startswith('_'):  # Skip private fields
                generator.create_section(key, value)
        
        # Save the PDF
        generator.pdf.output(output_file)
        print(f"PDF successfully generated: {output_file}")
        
    except json.JSONDecodeError as e:
        print(f"Error parsing JSON: {e}")
    except Exception as e:
        print(f"Error generating PDF: {e}")
        raise  # Re-raise the exception to see the full error trace

# Example usage:
if __name__ == "__main__":
    try:
        # Read JSON file with UTF-8 encoding
        with open('output/updated_resume.json', 'r', encoding='utf-8') as file:
            json_data = file.read()
        
        # Generate PDF
        json_to_pdf(json_data)
    except Exception as e:
        print(f"Error: {str(e)}")

PDF successfully generated: output.pdf


In [55]:
from fpdf import FPDF
import json
from typing import Dict, List, Any

class GenericPDFGenerator:
    def __init__(self, page_width: int = 210, page_height: int = 297):
        """Initialize the PDF generator with A4 size by default"""
        self.pdf = FPDF(format=(page_width, page_height))
        self.pdf.add_page()
        self.pdf.set_auto_page_break(auto=True, margin=15)
        self.pdf.set_font('Helvetica')
        self.left_margin = 20
        self.current_y = 20

    def format_key(self, key: str) -> str:
        """Format dictionary keys into readable titles"""
        return ' '.join(word.capitalize() for word in key.split('_')).strip()

    def clean_text(self, text: str) -> str:
        """Clean text by replacing problematic characters with ASCII equivalents"""
        chars_to_replace = {
            '"': '"',
            '"': '"',
            ''': "'",
            ''': "'",
            '–': '-',
            '—': '-',
            '•': '*',
            '…': '...',
            '\u2013': '-',
            '\u2014': '-',
            '\u2018': "'",
            '\u2019': "'",
            '\u201C': '"',
            '\u201D': '"',
            '\u2026': '...',
            '\u2022': '*',
            '\xa0': ' ',
        }
        
        for old, new in chars_to_replace.items():
            text = text.replace(old, new)
        return text.encode('ascii', 'replace').decode('ascii')

    def add_text(self, text: str, font_size: int = 10, spacing: int = 5, style: str = '') -> None:
        """Add text with specified font size and spacing"""
        self.pdf.set_font('Helvetica', style, font_size)
        self.pdf.set_xy(self.left_margin, self.current_y)
        clean_text = self.clean_text(str(text))
        self.pdf.multi_cell(0, spacing, clean_text)
        self.current_y = self.pdf.get_y() + spacing

    def format_experience(self, experience: Dict) -> None:
        """Format work experience entries"""
        # Company and Title
        self.pdf.set_font('Helvetica', 'B', 12)
        company_text = f"{experience.get('company', '')} - {experience.get('title', '')}"
        self.pdf.set_xy(self.left_margin, self.current_y)
        self.pdf.cell(0, 5, company_text)
        
        # Date and Location
        self.pdf.set_font('Helvetica', '', 10)
        date_location = f"{experience.get('date', '')} {experience.get('location', '')}"
        self.pdf.set_xy(self.left_margin + 120, self.current_y)
        self.pdf.cell(0, 5, date_location)
        self.current_y += 8

        # Bullet points
        if 'highlights' in experience:
            for highlight in experience['highlights']:
                self.pdf.set_xy(self.left_margin + 5, self.current_y)
                self.pdf.multi_cell(0, 5, f"• {highlight}")
                self.current_y = self.pdf.get_y() + 2

        self.current_y += 5

    def format_education(self, education: Dict) -> None:
        """Format education entries"""
        # School and Degree
        self.pdf.set_font('Helvetica', 'B', 12)
        school_text = f"{education.get('school', '')} - {education.get('degree', '')}"
        self.pdf.set_xy(self.left_margin, self.current_y)
        self.pdf.cell(0, 5, school_text)
        
        # Date and Location
        self.pdf.set_font('Helvetica', '', 10)
        date_location = f"{education.get('date', '')} {education.get('location', '')}"
        self.pdf.set_xy(self.left_margin + 120, self.current_y)
        self.pdf.cell(0, 5, date_location)
        self.current_y += 10

    def format_skills(self, skills: List[str]) -> None:
        """Format skills section"""
        self.pdf.set_font('Helvetica', '', 10)
        skills_text = '; '.join(skills)
        self.pdf.set_xy(self.left_margin, self.current_y)
        self.pdf.multi_cell(0, 5, skills_text)
        self.current_y = self.pdf.get_y() + 5

    def create_section(self, key: str, value: Any) -> None:
        """Create a section for each top-level key in the JSON"""
        # Special handling for name and title
        if key.lower() == 'name':
            self.add_text(value, 24, style='B')
            return
        elif key.lower() == 'title':
            self.add_text(value, 16)
            return
        elif key.lower() == 'contact':
            self.format_contact(value)
            return

        # Add section header
        self.current_y += 5
        self.add_text(self.format_key(key), 14, style='B')
        self.current_y += 2

        # Format section content based on type
        if key.lower() == 'experience':
            for exp in value:
                self.format_experience(exp)
        elif key.lower() == 'education':
            for edu in value:
                self.format_education(edu)
        elif key.lower() == 'skills':
            if isinstance(value, list):
                self.format_skills(value)
            else:
                self.format_skills(value.split(';'))
        else:
            self.format_value(value)

        self.current_y += 5

    def format_contact(self, contact: Dict) -> None:
        """Format contact information"""
        contact_items = []
        for key, value in contact.items():
            if value and not key.startswith('_'):
                contact_items.append(str(value))
        
        contact_text = '\n'.join(contact_items)
        self.add_text(contact_text, 10)

    def format_value(self, value: Any, indent_level: int = 0) -> None:
        """Format any other value types"""
        indent = self.left_margin + (indent_level * 10)
        if isinstance(value, (str, int, float)):
            self.pdf.set_xy(indent, self.current_y)
            self.add_text(str(value), 10)
        elif isinstance(value, (list, dict)):
            self.pdf.set_xy(indent, self.current_y)
            self.add_text(str(value), 10)

def json_to_pdf(json_data: str, output_file: str = "output.pdf") -> None:
    """Convert JSON data to PDF format"""
    try:
        data = json.loads(json_data)
        generator = GenericPDFGenerator()
        
        # Process each top-level key in the JSON
        for key, value in data.items():
            if not key.startswith('_'):
                generator.create_section(key, value)
        
        generator.pdf.output(output_file)
        print(f"PDF successfully generated: {output_file}")
        
    except json.JSONDecodeError as e:
        print(f"Error parsing JSON: {e}")
    except Exception as e:
        print(f"Error generating PDF: {e}")
        raise

if __name__ == "__main__":
    try:
        with open('output/updated_resume.json', 'r', encoding='utf-8') as file:
            json_data = file.read()
        json_to_pdf(json_data)
    except Exception as e:
        print(f"Error: {str(e)}")

PDF successfully generated: output.pdf


In [54]:
def create_resume_review_agent():

    Info_extract_agent = Agent(
        role='Resume information extracter',
        goal='Extract information from the provided resume files including summary, work experience, education, skills, name, address, email address, and phone number',
        backstory="""You are a skilled information extraction agent with expertise in parsing and analyzing resume data.""",
        verbose=True,
        allow_delegation=False,
        tools=[MDXSearchTool(mdx=resume_output_md_path)],
        llm=llm_35_turbo,
        )
    code_agent = Agent(
                role='Coder',
                goal='Write the python code to combine all the information',
                backstory="""You are a skilled coder with expertise in writing Python code to combine and format resume data.""",
                verbose=True,
                allow_delegation=False,
                # tools=[PDFSearchTool(pdf=sample_pdf_path)],
                llm=llm_35_turbo,
                )
    Info_extract_task = Task(
                description= "Extract information from the provided resume files including summary, work experience, education, skills, address, email address, and phone number",
                agent=Info_extract_agent,
                expected_output="Extracted information from the resume files",
                # output_file='output/updated_resume.pdf',
                )
    code_task = Task(
                description= "Write the python code to combine summary, work experience, education, skills, address, email address, and phone number to the pdf format similar to the sample resume",
                agent=code_agent,
                expected_output="Python code to combine and format resume data",
                output_file='output/code.txt',
                )
    resume_review_crew = Crew(
    agents=[Info_extract_agent, code_agent],
    tasks=[Info_extract_task, code_task],
    process=Process.sequential,
    manager_llm = manager_llm_35_turbo, # Assign the manager_llm to the Crew
    verbose=True
    )
    return resume_review_crew

# if __name__ == "main":
resume_output_md_path = 'output/updated_resume.md'
# sample_pdf_path = 'output/amazon-data-science-resume-example.pdf'
create_resume_review_agent().kickoff()

2024-11-18 21:01:56,406 - 8482288192 - __init__.py-__init__:531 - WARNING: Overriding of current TracerProvider is not allowed


 [DEBUG]: == Working Agent: Resume information extracter
 [INFO]: == Starting Task: Extract information from the provided resume files including summary, work experience, education, skills, address, email address, and phone number


> Entering new CrewAgentExecutor chain...
I need to start extracting the information from the resume files using the tools available.

Action: Search a MDX's content
Action Input: {"search_query": "summary"} 

Relevant Content:
3-hour reduction in query execution times. Automated data ingestion processes using AWS Glue, reducing manual effort by 32%. Conducted sentiment analysis on customer reviews using NLTK, providing valuable insights to the marketing team. eBay-Trainee Data Analyst 2012 - 2015 San Jose, CA Set up Kafka clusters and integrated data sources, resulting in a 30% improvement in data processing efﬁciency. Achieved a $4K reduction in infrastructure costs by containerizing data processing components. Spearheaded automated deployment scripts and

'```python\nfrom fpdf import FPDF\n\n# Create instance of FPDF class\npdf = FPDF()\npdf.set_auto_page_break(auto=True, margin=15)\npdf.add_page()\n\n# Set font for the entire pdf\npdf.set_font("Arial", size=12)\n\n# Add Summary\nsummary = "3-hour reduction in query execution times. Automated data ingestion processes using AWS Glue, reducing manual effort by 32%. Conducted sentiment analysis on customer reviews using NLTK, providing valuable insights to the marketing team. eBay-Trainee Data Analyst 2012 - 2015 San Jose, CA Set up Kafka clusters and integrated data sources, resulting in a 30% improvement in data processing efﬁciency. Achieved a $4K reduction in infrastructure costs by containerizing data processing components. Spearheaded automated deployment scripts and version control using Git, resulting in a 27% decrease in deployment errors. Used Python and SQL to clean and preprocess data, achieving a data quality improvement of 18%."\npdf.multi_cell(0, 10, summary)\npdf.ln(10)  # 

In [53]:
from fpdf import FPDF
import json
from typing import Dict, List, Any

class CompactResumePDFGenerator:
    def __init__(self):
        """Initialize the PDF generator with A4 size"""
        self.pdf = FPDF(format='A4')
        self.pdf.add_page()
        self.pdf.set_auto_page_break(auto=True, margin=15)
        self.pdf.set_font('Helvetica')
        self.left_margin = 10  # Reduced margin for more compact layout
        self.right_margin = 10
        self.current_y = 10
        self.page_width = 210
        self.content_width = self.page_width - self.left_margin - self.right_margin

    def clean_text(self, text: str) -> str:
        """Clean text by replacing problematic characters"""
        chars_to_replace = {
            '"': '"', '"': '"', ''': "'", ''': "'", '–': '-',
            '—': '-', '•': '*', '…': '...', '\u2013': '-',
            '\u2014': '-', '\u2018': "'", '\u2019': "'",
            '\u201C': '"', '\u201D': '"', '\u2026': '...',
            '\u2022': '*', '\xa0': ' '
        }
        for old, new in chars_to_replace.items():
            text = text.replace(old, new)
        return text.encode('ascii', 'replace').decode('ascii')

    def add_header(self, name: str, title: str) -> None:
        """Add header section with name and title"""
        self.pdf.set_xy(self.left_margin, self.current_y)
        self.pdf.set_font('Helvetica', 'B', 16)
        self.pdf.cell(0, 8, self.clean_text(name), ln=True)
        
        self.current_y = self.pdf.get_y()
        self.pdf.set_xy(self.left_margin, self.current_y)
        self.pdf.set_font('Helvetica', '', 12)
        self.pdf.cell(0, 6, self.clean_text(title), ln=True)
        self.current_y = self.pdf.get_y() + 2

    def add_contact(self, contact: Dict[str, str]) -> None:
        """Add contact information"""
        self.pdf.set_font('Helvetica', '', 10)
        contact_text = f"{contact['email']} | {contact['phone']} | {contact['location']}"
        self.pdf.set_xy(self.left_margin, self.current_y)
        self.pdf.cell(0, 5, self.clean_text(contact_text), ln=True)
        self.current_y = self.pdf.get_y() + 2

    def add_summary(self, summary: str) -> None:
        """Add professional summary"""
        self.pdf.set_font('Helvetica', '', 10)
        self.pdf.set_xy(self.left_margin, self.current_y)
        self.pdf.multi_cell(0, 5, self.clean_text(summary))
        self.current_y = self.pdf.get_y() + 2

    def add_section_header(self, title: str) -> None:
        """Add section header with line"""
        self.current_y += 2
        self.pdf.set_xy(self.left_margin, self.current_y)
        self.pdf.set_font('Helvetica', 'B', 12)
        self.pdf.cell(0, 6, self.clean_text(title), ln=True)
        
        # Add horizontal line
        self.current_y = self.pdf.get_y() - 2
        self.pdf.line(self.left_margin, self.current_y, 
                     self.page_width - self.right_margin, self.current_y)
        self.current_y += 3

    def add_experience(self, experiences: List[Dict[str, Any]]) -> None:
        """Add experience section"""
        self.add_section_header("EXPERIENCE")
        
        for exp in experiences:
            # Company and Title
            self.pdf.set_font('Helvetica', 'B', 10)
            company_text = f"{exp['company']} - {exp['title']}"
            self.pdf.set_xy(self.left_margin, self.current_y)
            self.pdf.cell(0, 5, self.clean_text(company_text), ln=True)
            
            # Period and Location
            self.pdf.set_font('Helvetica', '', 10)
            period_text = f"{exp['period']} {exp['location']}"
            self.pdf.set_xy(self.left_margin, self.pdf.get_y())
            self.pdf.cell(0, 5, self.clean_text(period_text), ln=True)
            
            # Bullets
            self.pdf.set_font('Helvetica', '', 10)
            for bullet in exp['bullets']:
                bullet_text = "* " + bullet
                self.pdf.set_xy(self.left_margin + 5, self.pdf.get_y())
                self.pdf.multi_cell(self.content_width - 5, 5, self.clean_text(bullet_text))
            
            self.current_y = self.pdf.get_y() + 2

    def add_education(self, education: List[Dict[str, Any]]) -> None:
        """Add education section"""
        self.add_section_header("EDUCATION")
        
        for edu in education:
            # School and Degree
            self.pdf.set_font('Helvetica', 'B', 10)
            school_text = f"{edu['school']}"
            self.pdf.set_xy(self.left_margin, self.current_y)
            self.pdf.cell(0, 5, self.clean_text(school_text), ln=True)
            
            # Degree, Period, and Location
            self.pdf.set_font('Helvetica', '', 10)
            details_text = f"{edu['degree']} | {edu['period']} | {edu['location']}"
            self.pdf.set_xy(self.left_margin, self.pdf.get_y())
            self.pdf.cell(0, 5, self.clean_text(details_text), ln=True)
            self.current_y = self.pdf.get_y() + 2

    def add_skills(self, skills: List[str]) -> None:
        """Add skills section"""
        self.add_section_header("SKILLS")
        
        self.pdf.set_font('Helvetica', '', 10)
        skills_text = "; ".join(skills)
        self.pdf.set_xy(self.left_margin, self.current_y)
        self.pdf.multi_cell(self.content_width, 5, self.clean_text(skills_text))
        self.current_y = self.pdf.get_y() + 2

def generate_compact_resume(json_data: str, output_file: str = "resume.pdf") -> None:
    """Generate a compact resume PDF from JSON data"""
    try:
        # Parse JSON data
        data = json.loads(json_data)
        
        # Initialize PDF generator
        generator = CompactResumePDFGenerator()
        
        # Add sections
        generator.add_header(data['name'], data['title'])
        generator.add_contact(data['contact'])
        generator.add_summary(data['summary'])
        generator.add_experience(data['experience'])
        generator.add_education(data['education'])
        generator.add_skills(data['skills'])
        
        # Save the PDF
        generator.pdf.output(output_file)
        print(f"PDF successfully generated: {output_file}")
        
    except Exception as e:
        print(f"Error generating PDF: {str(e)}")
        raise

# Example usage:
if __name__ == "__main__":
    try:
        # Read JSON file
        with open('output/updated_resume.json', 'r', encoding='utf-8') as file:
            json_data = file.read()
        
        # Generate PDF
        generate_compact_resume(json_data)
    except Exception as e:
        print(f"Error: {str(e)}")

Error generating PDF: 'location'
Error: 'location'


In [6]:
from fpdf import FPDF
import re

class UTF8PDF(FPDF):
    def __init__(self):
        super().__init__()
        # Add Unicode font support
        self.add_font('DejaVu', '', 'DejaVuSansCondensed.ttf', uni=True)
        self.add_font('DejaVu', 'B', 'DejaVuSansCondensed-Bold.ttf', uni=True)

    def clean_text(self, text):
        # Replace problematic characters
        text = text.replace('\u2013', '-')  # Replace en dash with hyphen
        text = text.replace('\u2014', '-')  # Replace em dash with hyphen
        text = text.replace('\u2018', "'")  # Replace curly single quotes
        text = text.replace('\u2019', "'")
        text = text.replace('\u201C', '"')  # Replace curly double quotes
        text = text.replace('\u201D', '"')
        return text

# Create instance of modified PDF class
pdf = FPDF()
pdf.set_auto_page_break(auto=True, margin=15)
pdf.add_page()

# Function to safely add text
def safe_text(text):
    return text.encode('latin-1', 'replace').decode('latin-1')

# Add a title
pdf.set_font("Arial", style="B", size=16)
pdf.cell(200, 10, "Resume", ln=True, align="C")

# Add summary section
pdf.set_font("Arial", style="B", size=14)
pdf.cell(200, 10, "Summary:", ln=True)
summary = """
3-hour reduction in query execution times. Automated data ingestion processes using AWS Glue,
reducing manual effort by 32%. Conducted sentiment analysis on customer reviews using NLTK,
providing valuable insights to the marketing team.
"""
pdf.set_font("Arial", size=12)
pdf.multi_cell(0, 10, safe_text(summary))

# Add work experience section
pdf.set_font("Arial", style="B", size=14)
pdf.cell(200, 10, "Work Experience:", ln=True)
work_experience = """
1. eBay-Trainee Data Analyst 2012 - 2015
- Set up Kafka clusters and integrated data sources, resulting in a 30% improvement in data processing efficiency.
- Achieved a $4K reduction in infrastructure costs by containerizing data processing components.
- Spearheaded automated deployment scripts and version control using Git, resulting in a 27% decrease in deployment errors.
- Used Python and SQL to clean and preprocess data, achieving a data quality improvement of 18%.
2. Fannie Mae Lead Quantitative Modeler June 2020 - August 2022
- Applied NLP techniques for domain-specific word embedding and NER using BERT/GPT-3 transfer learning.
- Enhanced document search and recommendations via text summarization and similarity.
- Generated a scoring algorithm to reflect the quality of review reports.
3. The Gallup Organization Lead Data Scientist July 2012 - June 2020
"""
pdf.set_font("Arial", size=12)
pdf.multi_cell(0, 10, safe_text(work_experience))

# Add education section
pdf.set_font("Arial", style="B", size=14)
pdf.cell(200, 10, "Education:", ln=True)
education = """
- Bachelor of Science, Computer Science, Stanford University, 2008 - 2012, Stanford, CA
- Bachelor of Science in Mathematical Statistics, Capital University of Economics and Business (Beijing, CHINA), Jul 2010, GPA: 3.93, Rank: 1/120
"""
pdf.set_font("Arial", size=12)
pdf.multi_cell(0, 10, safe_text(education))

# Add skills section
pdf.set_font("Arial", style="B", size=14)
pdf.cell(200, 10, "Skills:", ln=True)
skills = """
Python; Pandas; TensorFlow; Apache Hadoop; Amazon Redshift; AWS; NLTK; Apache Kafka; Git; Docker; SQL; LangChain;
crewAI; PyTorch; Keras; PySpark; H2O; Apache Spark; SAS; Tableau
"""
pdf.set_font("Arial", size=12)
pdf.multi_cell(0, 10, safe_text(skills))

# Add address section
pdf.set_font("Arial", style="B", size=14)
pdf.cell(200, 10, "Address:", ln=True)
address = "Washington D.C."
pdf.set_font("Arial", size=12)
pdf.cell(200, 10, safe_text(address), ln=True)

# Add email section
pdf.set_font("Arial", style="B", size=14)
pdf.cell(200, 10, "Email:", ln=True)
email = "[email protected]"
pdf.set_font("Arial", size=12)
pdf.cell(200, 10, safe_text(email), ln=True)

# Output the PDF file
pdf.output("resume.pdf")

''

In [7]:
from fpdf import FPDF

class PDF(FPDF):
    def header(self):
        # Empty header
        pass

    def footer(self):
        # Empty footer
        pass

# Create PDF instance
pdf = PDF()
pdf.add_page()
pdf.set_margins(20, 20, 20)  # Left, Top, Right margins
pdf.set_auto_page_break(auto=True, margin=15)

# Name - Large and Bold
pdf.set_font("Arial", "B", 24)
pdf.cell(0, 10, "Emma Davis", ln=True)

# Title - Slightly smaller, still bold
pdf.set_font("Arial", "B", 16)
pdf.cell(0, 10, "Amazon Data Scientist", ln=True)

# Add some space
pdf.ln(5)

# Summary - Regular text
pdf.set_font("Arial", size=11)
summary = """Dynamic data scientist with a strong foundation in machine learning, data analysis, and problem-solving. Eager to join Amazon's world-class data science team to leverage data-driven insights that drive business growth."""
pdf.multi_cell(0, 5, summary)

# Add some space
pdf.ln(5)

# Contact Information - Two columns
pdf.set_font("Arial", size=11)
left_col = """e.davis@email.com
(123) 456-7890"""
right_col = """San Jose, CA
LinkedIn"""

# Split contact info into two columns
original_x = pdf.get_x()
original_y = pdf.get_y()
pdf.multi_cell(90, 5, left_col)
pdf.set_xy(original_x + 100, original_y)
pdf.multi_cell(90, 5, right_col)

# Add some space
pdf.ln(10)

# Work Experience Section
pdf.set_font("Arial", "B", 14)
pdf.cell(0, 10, "Work Experience", ln=True)
pdf.ln(2)

# Adobe Experience
pdf.set_font("Arial", "B", 11)
pdf.cell(140, 5, "Adobe - Data Scientist")
pdf.set_font("Arial", "", 11)
pdf.cell(0, 5, "2018 - current", ln=True)
pdf.cell(0, 5, "San Jose, CA", ln=True)
pdf.ln(2)
adobe_exp = [
    "Led data analysis initiatives that resulted in a 37% increase in customer retention rates.",
    "Developed predictive models using TensorFlow, reducing forecasting errors by 21%.",
    "Implemented Apache Hadoop to analyze large-scale datasets, improving data processing speed by 33%.",
    "Utilized Pandas and Python for data manipulation, resulting in a 2-hour reduction in data cleaning time."
]
for bullet in adobe_exp:
    pdf.cell(10, 5, chr(127), ln=0)  # Bullet point
    pdf.multi_cell(0, 5, bullet)
pdf.ln(5)

# Cisco Experience
pdf.set_font("Arial", "B", 11)
pdf.cell(140, 5, "Cisco Systems - Junior Data Engineer")
pdf.set_font("Arial", "", 11)
pdf.cell(0, 5, "2015 - 2018", ln=True)
pdf.cell(0, 5, "San Jose, CA", ln=True)
pdf.ln(2)
cisco_exp = [
    "Collaborated with a cross-functional team to develop ETL pipelines, improving data processing efficiency by 26%.",
    "Leveraged Amazon Redshift to optimize data warehouse performance, resulting in a 3-hour reduction in query execution times.",
    "Automated data ingestion processes using AWS Glue, reducing manual effort by 32%.",
    "Conducted sentiment analysis on customer reviews using NLTK, providing valuable insights to the marketing team."
]
for bullet in cisco_exp:
    pdf.cell(10, 5, chr(127), ln=0)  # Bullet point
    pdf.multi_cell(0, 5, bullet)
pdf.ln(5)

# eBay Experience
pdf.set_font("Arial", "B", 11)
pdf.cell(140, 5, "eBay - Trainee Data Analyst")
pdf.set_font("Arial", "", 11)
pdf.cell(0, 5, "2012 - 2015", ln=True)
pdf.cell(0, 5, "San Jose, CA", ln=True)
pdf.ln(2)
ebay_exp = [
    "Set up Kafka clusters and integrated data sources, resulting in a 30% improvement in data processing efficiency.",
    "Achieved a $4K reduction in infrastructure costs by containerizing data processing components.",
    "Spearheaded automated deployment scripts and version control using Git, resulting in a 27% decrease in deployment errors.",
    "Used Python and SQL to clean and preprocess data, achieving a data quality improvement of 18%."
]
for bullet in ebay_exp:
    pdf.cell(10, 5, chr(127), ln=0)  # Bullet point
    pdf.multi_cell(0, 5, bullet)
pdf.ln(5)

# Education Section
pdf.set_font("Arial", "B", 14)
pdf.cell(0, 10, "Education", ln=True)
pdf.ln(2)

pdf.set_font("Arial", "B", 11)
pdf.cell(140, 5, "Stanford University - Bachelor of Science, Computer Science")
pdf.set_font("Arial", "", 11)
pdf.cell(0, 5, "2008 - 2012", ln=True)
pdf.cell(0, 5, "Stanford, CA", ln=True)
pdf.ln(5)

# Skills Section
pdf.set_font("Arial", "B", 14)
pdf.cell(0, 10, "Skills", ln=True)
pdf.ln(2)

pdf.set_font("Arial", "", 11)
skills = "Python; Pandas; TensorFlow; Apache Hadoop; Amazon Redshift; AWS; NLTK; Apache Kafka; Git; Docker"
pdf.multi_cell(0, 5, skills)

# Output the PDF
pdf.output("improved_resume.pdf")

''

In [8]:
from fpdf import FPDF

class PDF(FPDF):
    def header(self):
        pass
    
    def footer(self):
        pass

# Create PDF instance with smaller margins
pdf = PDF()
pdf.add_page()
pdf.set_margins(15, 10, 15)  # Reduced margins (left, top, right)
pdf.set_auto_page_break(auto=True, margin=10)  # Reduced bottom margin

# Name - Slightly smaller font
pdf.set_font("Arial", "B", 20)  # Reduced from 24
pdf.cell(0, 8, "Emma Davis", ln=True)  # Reduced height from 10

# Title - Slightly smaller
pdf.set_font("Arial", "B", 14)  # Reduced from 16
pdf.cell(0, 6, "Amazon Data Scientist", ln=True)  # Reduced height

# Summary - Compact
pdf.ln(2)  # Reduced spacing
pdf.set_font("Arial", size=10)  # Reduced from 11
summary = """Dynamic data scientist with a strong foundation in machine learning, data analysis, and problem-solving. Eager to join Amazon's world-class data science team to leverage data-driven insights that drive business growth."""
pdf.multi_cell(0, 4, summary)  # Reduced line height

# Contact Information - Two columns with less spacing
pdf.ln(2)
pdf.set_font("Arial", size=10)
left_col = """e.davis@email.com
(123) 456-7890"""
right_col = """San Jose, CA
LinkedIn"""

# Split contact info into two columns
original_x = pdf.get_x()
original_y = pdf.get_y()
pdf.multi_cell(90, 4, left_col)
pdf.set_xy(original_x + 100, original_y)
pdf.multi_cell(90, 4, right_col)

# Work Experience Section
pdf.ln(4)  # Reduced spacing
pdf.set_font("Arial", "B", 12)  # Reduced from 14
pdf.cell(0, 6, "Work Experience", ln=True)

# Adobe Experience
pdf.set_font("Arial", "B", 10)
pdf.cell(140, 4, "Adobe - Data Scientist")
pdf.set_font("Arial", "", 10)
pdf.cell(0, 4, "2018 - current", ln=True)
pdf.cell(0, 4, "San Jose, CA", ln=True)
adobe_exp = [
    "Led data analysis initiatives that resulted in a 37% increase in customer retention rates.",
    "Developed predictive models using TensorFlow, reducing forecasting errors by 21%.",
    "Implemented Apache Hadoop to analyze large-scale datasets, improving data processing speed by 33%.",
    "Utilized Pandas and Python for data manipulation, resulting in a 2-hour reduction in data cleaning time."
]
for bullet in adobe_exp:
    pdf.cell(5, 4, chr(127), ln=0)
    pdf.multi_cell(0, 4, bullet)

# Cisco Experience
pdf.ln(2)  # Reduced spacing
pdf.set_font("Arial", "B", 10)
pdf.cell(140, 4, "Cisco Systems - Junior Data Engineer")
pdf.set_font("Arial", "", 10)
pdf.cell(0, 4, "2015 - 2018", ln=True)
pdf.cell(0, 4, "San Jose, CA", ln=True)
cisco_exp = [
    "Collaborated with a cross-functional team to develop ETL pipelines, improving data processing efficiency by 26%.",
    "Leveraged Amazon Redshift to optimize data warehouse performance, resulting in a 3-hour reduction in query execution times.",
    "Automated data ingestion processes using AWS Glue, reducing manual effort by 32%.",
    "Conducted sentiment analysis on customer reviews using NLTK, providing valuable insights to the marketing team."
]
for bullet in cisco_exp:
    pdf.cell(5, 4, chr(127), ln=0)
    pdf.multi_cell(0, 4, bullet)

# eBay Experience
pdf.ln(2)  # Reduced spacing
pdf.set_font("Arial", "B", 10)
pdf.cell(140, 4, "eBay - Trainee Data Analyst")
pdf.set_font("Arial", "", 10)
pdf.cell(0, 4, "2012 - 2015", ln=True)
pdf.cell(0, 4, "San Jose, CA", ln=True)
ebay_exp = [
    "Set up Kafka clusters and integrated data sources, resulting in a 30% improvement in data processing efficiency.",
    "Achieved a $4K reduction in infrastructure costs by containerizing data processing components.",
    "Spearheaded automated deployment scripts and version control using Git, resulting in a 27% decrease in deployment errors.",
    "Used Python and SQL to clean and preprocess data, achieving a data quality improvement of 18%."
]
for bullet in ebay_exp:
    pdf.cell(5, 4, chr(127), ln=0)
    pdf.multi_cell(0, 4, bullet)

# Education Section
pdf.ln(4)
pdf.set_font("Arial", "B", 12)
pdf.cell(0, 6, "Education", ln=True)

pdf.set_font("Arial", "B", 10)
pdf.cell(140, 4, "Stanford University - Bachelor of Science, Computer Science")
pdf.set_font("Arial", "", 10)
pdf.cell(0, 4, "2008 - 2012", ln=True)
pdf.cell(0, 4, "Stanford, CA", ln=True)

# Skills Section
pdf.ln(4)
pdf.set_font("Arial", "B", 12)
pdf.cell(0, 6, "Skills", ln=True)

pdf.set_font("Arial", "", 10)
skills = "Python; Pandas; TensorFlow; Apache Hadoop; Amazon Redshift; AWS; NLTK; Apache Kafka; Git; Docker"
pdf.multi_cell(0, 4, skills)

# Output the PDF
pdf.output("output/one_page_resume.pdf")

''

In [9]:
from fpdf import FPDF
import json

class ResumePDF(FPDF):
    def __init__(self):
        super().__init__()
        self.set_margins(15, 10, 15)
        self.set_auto_page_break(auto=True, margin=10)
    
    def header(self):
        pass
    
    def footer(self):
        pass
    
    def add_name_and_title(self, name, title):
        self.set_font("Arial", "B", 20)
        self.cell(0, 8, name, ln=True)
        self.set_font("Arial", "B", 14)
        self.cell(0, 6, title, ln=True)
    
    def add_summary(self, summary):
        self.ln(2)
        self.set_font("Arial", size=10)
        self.multi_cell(0, 4, summary)
    
    def add_contact_info(self, contact_info):
        self.ln(2)
        self.set_font("Arial", size=10)
        
        # Left column
        original_x = self.get_x()
        original_y = self.get_y()
        left_col = f"{contact_info['email']}\n{contact_info['phone']}"
        right_col = f"{contact_info['location']}\n{contact_info['linkedin']}"
        
        self.multi_cell(90, 4, left_col)
        self.set_xy(original_x + 100, original_y)
        self.multi_cell(90, 4, right_col)
    
    def add_section_header(self, title):
        self.ln(4)
        self.set_font("Arial", "B", 12)
        self.cell(0, 6, title, ln=True)
    
    def add_experience(self, experience):
        self.set_font("Arial", "B", 10)
        self.cell(140, 4, f"{experience['company']} - {experience['title']}")
        self.set_font("Arial", "", 10)
        self.cell(0, 4, experience['period'], ln=True)
        self.cell(0, 4, experience['location'], ln=True)
        
        for bullet in experience['bullets']:
            self.cell(5, 4, chr(127), ln=0)
            self.multi_cell(0, 4, bullet)
        self.ln(2)
    
    def add_education(self, education):
        self.set_font("Arial", "B", 10)
        self.cell(140, 4, education['school'])
        self.set_font("Arial", "", 10)
        self.cell(0, 4, education['period'], ln=True)
        self.cell(0, 4, education['location'], ln=True)
    
    def add_skills(self, skills):
        self.set_font("Arial", "", 10)
        self.multi_cell(0, 4, '; '.join(skills))
    
    def create_resume(self, resume_data):
        self.add_page()
        
        # Add name and title
        self.add_name_and_title(resume_data['name'], resume_data['title'])
        
        # Add summary
        self.add_summary(resume_data['summary'])
        
        # Add contact information
        self.add_contact_info(resume_data['contact'])
        
        # Add work experience
        self.add_section_header("Work Experience")
        for exp in resume_data['experience']:
            self.add_experience(exp)
        
        # Add education
        self.add_section_header("Education")
        self.add_education(resume_data['education'])
        
        # Add skills
        self.add_section_header("Skills")
        self.add_skills(resume_data['skills'])

# Resume content in JSON format
resume_content = {
    "name": "Emma Davis",
    "title": "Amazon Data Scientist",
    "summary": "Dynamic data scientist with a strong foundation in machine learning, data analysis, and problem-solving. Eager to join Amazon's world-class data science team to leverage data-driven insights that drive business growth.",
    "contact": {
        "email": "e.davis@email.com",
        "phone": "(123) 456-7890",
        "location": "San Jose, CA",
        "linkedin": "LinkedIn"
    },
    "experience": [
        {
            "company": "Adobe",
            "title": "Data Scientist",
            "period": "2018 - current",
            "location": "San Jose, CA",
            "bullets": [
                "Led data analysis initiatives that resulted in a 37% increase in customer retention rates.",
                "Developed predictive models using TensorFlow, reducing forecasting errors by 21%.",
                "Implemented Apache Hadoop to analyze large-scale datasets, improving data processing speed by 33%.",
                "Utilized Pandas and Python for data manipulation, resulting in a 2-hour reduction in data cleaning time."
            ]
        },
        {
            "company": "Cisco Systems",
            "title": "Junior Data Engineer",
            "period": "2015 - 2018",
            "location": "San Jose, CA",
            "bullets": [
                "Collaborated with a cross-functional team to develop ETL pipelines, improving data processing efficiency by 26%.",
                "Leveraged Amazon Redshift to optimize data warehouse performance, resulting in a 3-hour reduction in query execution times.",
                "Automated data ingestion processes using AWS Glue, reducing manual effort by 32%.",
                "Conducted sentiment analysis on customer reviews using NLTK, providing valuable insights to the marketing team."
            ]
        },
        {
            "company": "eBay",
            "title": "Trainee Data Analyst",
            "period": "2012 - 2015",
            "location": "San Jose, CA",
            "bullets": [
                "Set up Kafka clusters and integrated data sources, resulting in a 30% improvement in data processing efficiency.",
                "Achieved a $4K reduction in infrastructure costs by containerizing data processing components.",
                "Spearheaded automated deployment scripts and version control using Git, resulting in a 27% decrease in deployment errors.",
                "Used Python and SQL to clean and preprocess data, achieving a data quality improvement of 18%."
            ]
        }
    ],
    "education": {
        "school": "Stanford University - Bachelor of Science, Computer Science",
        "period": "2008 - 2012",
        "location": "Stanford, CA"
    },
    "skills": [
        "Python",
        "Pandas",
        "TensorFlow",
        "Apache Hadoop",
        "Amazon Redshift",
        "AWS",
        "NLTK",
        "Apache Kafka",
        "Git",
        "Docker"
    ]
}

# Create and save the resume
def create_resume_pdf(resume_data, output_file):
    resume = ResumePDF()
    resume.create_resume(resume_data)
    resume.output(output_file)

# Generate the PDF
create_resume_pdf(resume_content, "output/structured_resume.pdf")

In [10]:
from fpdf import FPDF
import json

class ResumePDF(FPDF):
    def __init__(self):
        super().__init__()
        # Smaller margins to maximize space
        self.set_margins(10, 8, 10)  # left, top, right
        self.set_auto_page_break(auto=True, margin=8)  # bottom margin
        
    def header(self):
        pass
    
    def footer(self):
        pass
    
    def calculate_content_height(self, resume_data):
        """Calculate approximate content height to adjust spacing"""
        total_lines = 0
        # Name and title (2 lines)
        total_lines += 2
        # Summary (estimate based on length)
        total_lines += len(resume_data['summary']) / 90  # characters per line
        # Contact info (2 lines)
        total_lines += 2
        # Experience (company + location + bullets)
        for exp in resume_data['experience']:
            total_lines += 2  # Company and location
            total_lines += len(exp['bullets'])
        # Education (2 lines)
        total_lines += 2
        # Skills (estimate based on length)
        total_lines += len('; '.join(resume_data['skills'])) / 90
        # Section headers (4 sections)
        total_lines += 4
        
        return total_lines * 4  # Approximate height in mm

    def add_name_and_title(self, name, title):
        # Larger font for name
        self.set_font("Arial", "B", 22)
        self.cell(0, 9, name, ln=True)
        # Smaller font for title
        self.set_font("Arial", "B", 16)
        self.cell(0, 7, title, ln=True)
    
    def add_summary(self, summary):
        self.ln(1)
        self.set_font("Arial", size=10)
        self.multi_cell(0, 4, summary)
    
    def add_contact_info(self, contact_info):
        self.ln(1)
        self.set_font("Arial", size=10)
        
        # Split contact info into two columns
        left_col = f"{contact_info['email']}\n{contact_info['phone']}"
        right_col = f"{contact_info['location']}\n{contact_info['linkedin']}"
        
        # Store current position
        original_x = self.get_x()
        original_y = self.get_y()
        
        # Left column
        self.multi_cell(95, 4, left_col)
        
        # Right column
        self.set_xy(original_x + 95, original_y)
        self.multi_cell(95, 4, right_col)
    
    def add_section_header(self, title, spacing=3):
        self.ln(spacing)
        self.set_font("Arial", "B", 14)
        self.cell(0, 6, title, ln=True)
    
    def add_experience(self, experience, spacing=2):
        # Company and title
        self.set_font("Arial", "B", 11)
        title_line = f"{experience['company']} - {experience['title']}"
        self.cell(140, 4, title_line)
        
        # Period
        self.set_font("Arial", "", 10)
        self.cell(0, 4, experience['period'], ln=True)
        
        # Location
        self.cell(0, 4, experience['location'], ln=True)
        
        # Bullets
        self.set_font("Arial", "", 10)
        for bullet in experience['bullets']:
            self.cell(5, 4, chr(127), ln=0)  # Bullet point
            self.multi_cell(0, 4, bullet)
        
        self.ln(spacing)
    
    def add_education(self, education):
        self.set_font("Arial", "B", 11)
        self.cell(140, 4, education['school'])
        self.set_font("Arial", "", 10)
        self.cell(0, 4, education['period'], ln=True)
        self.cell(0, 4, education['location'], ln=True)
    
    def add_skills(self, skills):
        self.set_font("Arial", "", 10)
        skills_text = '; '.join(skills)
        self.multi_cell(0, 4, skills_text)
    
    def create_resume(self, resume_data):
        # Add the page
        self.add_page()
        
        # Calculate total content height
        content_height = self.calculate_content_height(resume_data)
        page_height = 297  # A4 height in mm
        available_height = page_height - (self.t_margin + self.b_margin)
        
        # Add name and title
        self.add_name_and_title(resume_data['name'], resume_data['title'])
        
        # Add summary
        self.add_summary(resume_data['summary'])
        
        # Add contact information
        self.add_contact_info(resume_data['contact'])
        
        # Adjust section spacing based on content
        section_spacing = min(6, max(3, (available_height - content_height) / 5))
        
        # Add work experience
        self.add_section_header("Work Experience", section_spacing)
        for i, exp in enumerate(resume_data['experience']):
            # Less spacing for last experience entry
            spacing = section_spacing if i < len(resume_data['experience'])-1 else 2
            self.add_experience(exp, spacing)
        
        # Add education
        self.add_section_header("Education", section_spacing)
        self.add_education(resume_data['education'])
        
        # Add skills
        self.add_section_header("Skills", section_spacing)
        self.add_skills(resume_data['skills'])

def create_resume_pdf(resume_data, output_file):
    resume = ResumePDF()
    resume.create_resume(resume_data)
    resume.output(output_file)

# Resume content in JSON format
resume_content = {
    "name": "Emma Davis",
    "title": "Amazon Data Scientist",
    "summary": "Dynamic data scientist with a strong foundation in machine learning, data analysis, and problem-solving. Eager to join Amazon's world-class data science team to leverage data-driven insights that drive business growth.",
    "contact": {
        "email": "e.davis@email.com",
        "phone": "(123) 456-7890",
        "location": "San Jose, CA",
        "linkedin": "LinkedIn"
    },
    "experience": [
        {
            "company": "Adobe",
            "title": "Data Scientist",
            "period": "2018 - current",
            "location": "San Jose, CA",
            "bullets": [
                "Led data analysis initiatives that resulted in a 37% increase in customer retention rates.",
                "Developed predictive models using TensorFlow, reducing forecasting errors by 21%.",
                "Implemented Apache Hadoop to analyze large-scale datasets, improving data processing speed by 33%.",
                "Utilized Pandas and Python for data manipulation, resulting in a 2-hour reduction in data cleaning time."
            ]
        },
        {
            "company": "Cisco Systems",
            "title": "Junior Data Engineer",
            "period": "2015 - 2018",
            "location": "San Jose, CA",
            "bullets": [
                "Collaborated with a cross-functional team to develop ETL pipelines, improving data processing efficiency by 26%.",
                "Leveraged Amazon Redshift to optimize data warehouse performance, resulting in a 3-hour reduction in query execution times.",
                "Automated data ingestion processes using AWS Glue, reducing manual effort by 32%.",
                "Conducted sentiment analysis on customer reviews using NLTK, providing valuable insights to the marketing team."
            ]
        },
        {
            "company": "eBay",
            "title": "Trainee Data Analyst",
            "period": "2012 - 2015",
            "location": "San Jose, CA",
            "bullets": [
                "Set up Kafka clusters and integrated data sources, resulting in a 30% improvement in data processing efficiency.",
                "Achieved a $4K reduction in infrastructure costs by containerizing data processing components.",
                "Spearheaded automated deployment scripts and version control using Git, resulting in a 27% decrease in deployment errors.",
                "Used Python and SQL to clean and preprocess data, achieving a data quality improvement of 18%."
            ]
        }
    ],
    "education": {
        "school": "Stanford University - Bachelor of Science, Computer Science",
        "period": "2008 - 2012",
        "location": "Stanford, CA"
    },
    "skills": [
        "Python",
        "Pandas",
        "TensorFlow",
        "Apache Hadoop",
        "Amazon Redshift",
        "AWS",
        "NLTK",
        "Apache Kafka",
        "Git",
        "Docker"
    ]
}

# Generate the PDF
create_resume_pdf(resume_content, "output/full_page_resume.pdf")

In [11]:
from fpdf import FPDF
import json

class ResumePDF(FPDF):
    def __init__(self):
        super().__init__()
        # Even smaller margins
        self.set_margins(8, 6, 8)  # left, top, right
        self.set_auto_page_break(auto=True, margin=6)  # bottom margin
        
    def header(self):
        pass
    
    def footer(self):
        pass
    
    def get_effective_page_height(self):
        """Get available page height for content"""
        return self.h - self.t_margin - self.b_margin
    
    def calculate_spacing(self, total_sections):
        """Calculate dynamic spacing based on available space"""
        available_height = self.get_effective_page_height()
        return available_height / (total_sections * 4)  # Divide space among sections
    
    def add_name_and_title(self, name, title):
        self.set_font("Arial", "B", 24)  # Increased name size
        self.cell(0, 10, name, ln=True)
        self.set_font("Arial", "B", 16)
        self.cell(0, 8, title, ln=True)
    
    def add_summary(self, summary):
        self.set_font("Arial", size=10)
        self.multi_cell(0, 5, summary)
    
    def add_contact_info(self, contact_info):
        self.ln(1)
        self.set_font("Arial", size=10)
        
        # Store current position
        original_x = self.get_x()
        original_y = self.get_y()
        
        # Left column
        left_col = f"{contact_info['email']}\n{contact_info['phone']}"
        self.multi_cell(95, 5, left_col)
        
        # Right column
        self.set_xy(original_x + 95, original_y)
        right_col = f"{contact_info['location']}\n{contact_info['linkedin']}"
        self.multi_cell(95, 5, right_col)
        self.ln(2)
    
    def add_section_header(self, title):
        self.set_font("Arial", "B", 14)
        self.ln(4)  # Add some space before section
        self.cell(0, 8, title, ln=True)  # Increased height
        self.ln(2)  # Add space after header
    
    def add_experience(self, experience, is_last=False):
        # Company and title with more space
        self.set_font("Arial", "B", 11)
        title_line = f"{experience['company']} - {experience['title']}"
        self.cell(140, 6, title_line)  # Increased height
        
        # Period
        self.set_font("Arial", "", 10)
        self.cell(0, 6, experience['period'], ln=True)
        
        # Location
        self.cell(0, 5, experience['location'], ln=True)
        
        # Add some space before bullets
        self.ln(1)
        
        # Bullets with more spacing
        self.set_font("Arial", "", 10)
        for bullet in experience['bullets']:
            self.cell(5, 5, chr(127), ln=0)  # Bullet point
            self.multi_cell(0, 5, bullet)  # Increased line height
        
        # Add space after experience section
        if not is_last:
            self.ln(4)
    
    def add_education(self, education):
        self.set_font("Arial", "B", 11)
        self.cell(140, 6, education['school'])
        self.set_font("Arial", "", 10)
        self.cell(0, 6, education['period'], ln=True)
        self.cell(0, 5, education['location'], ln=True)
        self.ln(2)
    
    def add_skills(self, skills):
        self.set_font("Arial", "", 10)
        # Format skills with more spacing
        skills_text = '; '.join(skills)
        # Split skills into multiple lines for better distribution
        words_per_line = len(skills) // 3  # Distribute across ~3 lines
        formatted_skills = []
        current_line = []
        
        for skill in skills:
            current_line.append(skill)
            if len(current_line) >= words_per_line:
                formatted_skills.append('; '.join(current_line))
                current_line = []
        
        if current_line:
            formatted_skills.append('; '.join(current_line))
            
        for line in formatted_skills:
            self.multi_cell(0, 6, line)  # Increased line height
    
    def create_resume(self, resume_data):
        self.add_page()
        
        # Add name and title
        self.add_name_and_title(resume_data['name'], resume_data['title'])
        self.ln(2)
        
        # Add summary
        self.add_summary(resume_data['summary'])
        self.ln(3)
        
        # Add contact information
        self.add_contact_info(resume_data['contact'])
        
        # Add work experience
        self.add_section_header("Work Experience")
        for i, exp in enumerate(resume_data['experience']):
            is_last = i == len(resume_data['experience']) - 1
            self.add_experience(exp, is_last)
        
        # Add education with more spacing
        self.add_section_header("Education")
        self.add_education(resume_data['education'])
        
        # Add skills with more spacing
        self.add_section_header("Skills")
        self.add_skills(resume_data['skills'])

# Resume content remains the same as in your previous code
resume_content = {
    "name": "Emma Davis",
    "title": "Amazon Data Scientist",
    "summary": "Dynamic data scientist with a strong foundation in machine learning, data analysis, and problem-solving. Eager to join Amazon's world-class data science team to leverage data-driven insights that drive business growth.",
    "contact": {
        "email": "e.davis@email.com",
        "phone": "(123) 456-7890",
        "location": "San Jose, CA",
        "linkedin": "LinkedIn"
    },
    "experience": [
        {
            "company": "Adobe",
            "title": "Data Scientist",
            "period": "2018 - current",
            "location": "San Jose, CA",
            "bullets": [
                "Led data analysis initiatives that resulted in a 37% increase in customer retention rates.",
                "Developed predictive models using TensorFlow, reducing forecasting errors by 21%.",
                "Implemented Apache Hadoop to analyze large-scale datasets, improving data processing speed by 33%.",
                "Utilized Pandas and Python for data manipulation, resulting in a 2-hour reduction in data cleaning time."
            ]
        },
        {
            "company": "Cisco Systems",
            "title": "Junior Data Engineer",
            "period": "2015 - 2018",
            "location": "San Jose, CA",
            "bullets": [
                "Collaborated with a cross-functional team to develop ETL pipelines, improving data processing efficiency by 26%.",
                "Leveraged Amazon Redshift to optimize data warehouse performance, resulting in a 3-hour reduction in query execution times.",
                "Automated data ingestion processes using AWS Glue, reducing manual effort by 32%.",
                "Conducted sentiment analysis on customer reviews using NLTK, providing valuable insights to the marketing team."
            ]
        },
        {
            "company": "eBay",
            "title": "Trainee Data Analyst",
            "period": "2012 - 2015",
            "location": "San Jose, CA",
            "bullets": [
                "Set up Kafka clusters and integrated data sources, resulting in a 30% improvement in data processing efficiency.",
                "Achieved a $4K reduction in infrastructure costs by containerizing data processing components.",
                "Spearheaded automated deployment scripts and version control using Git, resulting in a 27% decrease in deployment errors.",
                "Used Python and SQL to clean and preprocess data, achieving a data quality improvement of 18%."
            ]
        }
    ],
    "education": {
        "school": "Stanford University - Bachelor of Science, Computer Science",
        "period": "2008 - 2012",
        "location": "Stanford, CA"
    },
    "skills": [
        "Python",
        "Pandas",
        "TensorFlow",
        "Apache Hadoop",
        "Amazon Redshift",
        "AWS",
        "NLTK",
        "Apache Kafka",
        "Git",
        "Docker"
    ]
}

def create_resume_pdf(resume_data, output_file):
    resume = ResumePDF()
    resume.create_resume(resume_data)
    resume.output(output_file)

# Generate the PDF
create_resume_pdf(resume_content, "output/full_page_resume.pdf")

In [ ]:
from fpdf import FPDF
import json

class ResumePDF(FPDF):
    def __init__(self):
        super().__init__()
        self.set_margins(8, 6, 8)
        self.set_auto_page_break(auto=False)
        self.page_height = 287
        
    def header(self):
        pass
    
    def footer(self):
        pass

    def estimate_content_height(self, resume_data):
        """Estimate content height without actually creating the PDF"""
        total_height = 0
        
        # Name and title
        total_height += 20
        
        # Summary
        total_height += len(resume_data['summary']) / 100 * 5
        
        # Contact info
        total_height += 15
        
        # Experience section
        total_height += 10
        for exp in resume_data['experience']:
            total_height += 15
            total_height += len(exp['bullets']) * 6
        
        # Education section
        total_height += 25
        
        # Skills section
        total_height += 20
        
        return total_height

    def distribute_skills(self, skills, max_lines=3):
        """Distribute skills evenly across lines without splitting words"""
        if not skills:
            return []
            
        total_chars = sum(len(skill) for skill in skills) + (len(skills) - 1) * 2  # Count for separators
        chars_per_line = total_chars // max_lines + 1
        
        lines = []
        current_line = []
        current_length = 0
        
        for skill in skills:
            # Length if we add this skill
            added_length = current_length + len(skill) + (2 if current_line else 0)  # +2 for "; "
            
            if current_line and added_length > chars_per_line and len(lines) < max_lines - 1:
                # Start new line if adding would exceed target length
                lines.append(current_line)
                current_line = [skill]
                current_length = len(skill)
            else:
                current_line.append(skill)
                current_length = added_length
        
        if current_line:
            lines.append(current_line)
        
        return lines

    def create_resume(self, resume_data):
        # Calculate scaling
        estimated_height = self.estimate_content_height(resume_data)
        scaling_factor = min(1.0, self.page_height / estimated_height)
        
        # Adjust font sizes
        name_size = max(16, min(24 * scaling_factor, 24))
        title_size = max(12, min(16 * scaling_factor, 16))
        base_size = max(9, min(11 * scaling_factor, 11))
        section_size = max(12, min(14 * scaling_factor, 14))
        
        self.add_page()
        
        # Name
        self.set_font("Arial", "B", int(name_size))
        self.cell(0, 8 * scaling_factor, resume_data['name'], ln=True)
        
        # Title
        self.set_font("Arial", "B", int(title_size))
        self.cell(0, 6 * scaling_factor, resume_data['title'], ln=True)
        
        # Summary
        self.ln(2 * scaling_factor)
        self.set_font("Arial", "", int(base_size))
        self.multi_cell(0, 4 * scaling_factor, resume_data['summary'])
        
        # Contact Info
        self.ln(2 * scaling_factor)
        contact = resume_data['contact']
        original_x = self.get_x()
        original_y = self.get_y()
        self.multi_cell(95, 4 * scaling_factor, f"{contact['email']}\n{contact['phone']}")
        self.set_xy(original_x + 95, original_y)
        self.multi_cell(95, 4 * scaling_factor, f"{contact['location']}\n{contact['linkedin']}")
        
        # Experience Section
        self.ln(4 * scaling_factor)
        self.set_font("Arial", "B", int(section_size))
        self.cell(0, 6 * scaling_factor, "Work Experience", ln=True)
        
        for exp in resume_data['experience']:
            self.ln(2 * scaling_factor)
            self.set_font("Arial", "B", int(base_size))
            title_line = f"{exp['company']} - {exp['title']}"
            self.cell(140, 5 * scaling_factor, title_line)
            self.set_font("Arial", "", int(base_size))
            self.cell(0, 5 * scaling_factor, exp['period'], ln=True)
            self.cell(0, 4 * scaling_factor, exp['location'], ln=True)
            
            self.ln(1 * scaling_factor)
            for bullet in exp['bullets']:
                self.cell(5, 4 * scaling_factor, chr(127), ln=0)
                self.multi_cell(0, 4 * scaling_factor, bullet)
            self.ln(1 * scaling_factor)
        
        # Education Section
        self.ln(3 * scaling_factor)
        self.set_font("Arial", "B", int(section_size))
        self.cell(0, 6 * scaling_factor, "Education", ln=True)
        
        edu = resume_data['education']
        self.ln(2 * scaling_factor)
        self.set_font("Arial", "B", int(base_size))
        self.cell(140, 5 * scaling_factor, edu['school'])
        self.set_font("Arial", "", int(base_size))
        self.cell(0, 5 * scaling_factor, edu['period'], ln=True)
        self.cell(0, 4 * scaling_factor, edu['location'], ln=True)
        
        # Skills Section
        self.ln(3 * scaling_factor)
        self.set_font("Arial", "B", int(section_size))
        self.cell(0, 6 * scaling_factor, "Skills", ln=True)
        
        self.ln(2 * scaling_factor)
        self.set_font("Arial", "", int(base_size))
        
        # Distribute skills across lines without splitting words
        skill_lines = self.distribute_skills(resume_data['skills'])
        for skills in skill_lines:
            skill_line = '; '.join(skills)
            self.multi_cell(0, 4 * scaling_factor, skill_line)

def create_resume_pdf(resume_data, output_file):
    resume = ResumePDF()
    resume.create_resume(resume_data)
    resume.output(output_file)

# Resume content (your existing content)
resume_content = {
  "name": "Diana Liu",
  "title": "AI Data Scientist",
  "contact": {
        "email": "lxjiao0805@gmail.com",
        "phone": "202.739.1368",
        "location": "Fairfax, VA",
        "linkedin": "LinkedIn"
    },
  "summary": "Proficient in mathematical statistics, econometrics, machine learning and deep learning. Offering 15 years of extensive project management expertise with a deep involvement in experimental design, data integration and cleansing, feature engineering, as well as mastery in machine learning, optimization, and deep learning implementations. With extensive experience in leveraging advanced AI technologies, I have successfully utilized prompt engineering, Retrieval-Augmented Generation (RAG), and agents to develop robust applications that address complex business challenges, automate processes, and drive business growth.",
  "experience": [
    {
      "title": "AI Data Scientist Freelancer",
      "duration": "March 2023 – present",
      "bullets": [
        "Enhance product descriptions and recommendations through various recommender systems.",
        "3-hour reduction in query execution times.",
        "Automated data ingestion processes using AWS Glue, reducing manual effort by 32%.",
        "Conducted sentiment analysis on customer reviews using NLTK, providing valuable insights to the marketing team."
      ]
    },
    {
      "title": "eBay-Trainee Data Analyst",
      "duration": "2012 - 2015",
      "location": "San Jose, CA",
      "bullets": [
        "Set up Kafka clusters and integrated data sources, resulting in a 30% improvement in data processing efficiency.",
        "Achieved a $4K reduction in infrastructure costs by containerizing data processing components.",
        "Spearheaded automated deployment scripts and version control using Git, resulting in a 27% decrease in deployment errors.",
        "Used Python and SQL to clean and preprocess data, achieving a data quality improvement of 18%."
      ]
    }
  ],
  "education": [
    {
      "degree": "Professional Degree in Artificial Intelligence",
      "date": "Dec 2022",
      "institution": "Stanford University"
    },
    {
      "degree": "Master of Science in Statistics",
      "date": "May 2012",
      "gPA": "3.89",
      "institution": "The George Washington University"
    },
    {
      "degree": "Bachelor of Science in Mathematical Statistics",
      "date": "Jul 2010",
      "gPA": "3.93",
      "rank": "1/120",
      "institution": "Capital University of Economics and Business (Beijing, CHINA)"
    }
  ],
  "skills": [
    "Python",
    "Pandas",
    "TensorFlow",
    "Apache Hadoop",
    "Amazon Redshift",
    "AWS",
    "NLTK",
    "Apache Kafka",
    "Git",
    "Docker"
  ]
}
# # resume_content = {
#     "name": "Emma Davis",
#     "title": "Amazon Data Scientist",
#     "summary": "Dynamic data scientist with a strong foundation in machine learning, data analysis, and problem-solving. Eager to join Amazon's world-class data science team to leverage data-driven insights that drive business growth.",
#     "contact": {
#         "email": "e.davis@email.com",
#         "phone": "(123) 456-7890",
#         "location": "San Jose, CA",
#         "linkedin": "LinkedIn"
#     },
#     "experience": [
#         {
#             "company": "Adobe",
#             "title": "Data Scientist",
#             "period": "2018 - current",
#             "location": "San Jose, CA",
#             "bullets": [
#                 "Led data analysis initiatives that resulted in a 37% increase in customer retention rates.",
#                 "Developed predictive models using TensorFlow, reducing forecasting errors by 21%.",
#                 "Implemented Apache Hadoop to analyze large-scale datasets, improving data processing speed by 33%.",
#                 "Utilized Pandas and Python for data manipulation, resulting in a 2-hour reduction in data cleaning time."
#             ]
#         },
#         {
#             "company": "Cisco Systems",
#             "title": "Junior Data Engineer",
#             "period": "2015 - 2018",
#             "location": "San Jose, CA",
#             "bullets": [
#                 "Collaborated with a cross-functional team to develop ETL pipelines, improving data processing efficiency by 26%.",
#                 "Leveraged Amazon Redshift to optimize data warehouse performance, resulting in a 3-hour reduction in query execution times.",
#                 "Automated data ingestion processes using AWS Glue, reducing manual effort by 32%.",
#                 "Conducted sentiment analysis on customer reviews using NLTK, providing valuable insights to the marketing team."
#             ]
#         },
#         {
#             "company": "eBay",
#             "title": "Trainee Data Analyst",
#             "period": "2012 - 2015",
#             "location": "San Jose, CA",
#             "bullets": [
#                 "Set up Kafka clusters and integrated data sources, resulting in a 30% improvement in data processing efficiency.",
#                 "Achieved a $4K reduction in infrastructure costs by containerizing data processing components.",
#                 "Spearheaded automated deployment scripts and version control using Git, resulting in a 27% decrease in deployment errors.",
#                 "Used Python and SQL to clean and preprocess data, achieving a data quality improvement of 18%."
#             ]
#         }
#     ],
#     "education": {
#         "school": "Stanford University - Bachelor of Science, Computer Science",
#         "period": "2008 - 2012",
#         "location": "Stanford, CA"
#     },
#     "skills": [
#         "Python",
#         "Pandas",
#         "TensorFlow",
#         "Apache Hadoop",
#         "Amazon Redshift",
#         "AWS",
#         "NLTK",
#         "Apache Kafka",
#         "Git",
#         "Docker"
#     ]
# }

# # Generate the PDF
create_resume_pdf(resume_content, "output/single_page_resume.pdf")

KeyError: 'company'

In [ ]:
# create 
def create_resume_review_agent():
    Extract_info_agent = Agent(
        role='Information Extraction Expert',
        goal='Extract relevant information from the provided resumes',
        backstory="""You are an expert in information extraction and analysis."""
        verbose=True,
        allow_delegation=False,
        llm=llm_35_turbo,
        tools=[MDXSearchTool(mdx = 'output/update_resume.md')],
        allow_delegation=False,
    )

    resume_review_agent = Agent(
        role='Resume Review and Rewrite Expert',
        goal='Provide comprehensive feedback on resumes and rewrite them based on suggestions',
        backstory="""You are an experienced resume expert with a keen eye for detail 
        and a deep understanding of various industries and job markets. Your expertise 
        helps job seekers create compelling resumes that stand out to potential employers and match job requirements.""",
        verbose=True,
        allow_delegation=False,
        tools=[ScrapeWebsiteTool(website_url=self.jd_url)],
        llm=llm_35_turbo,
    )

    review_task = Task(
        description="""Review the provided resumes and analyze their strengths and weaknesses. 
        Focus on structure, content, formatting, and overall impact. Note that the original 
        resume is in HTML format (converted from Markdown) and the updated resume is in plain text 
        (extracted from PDF). Consider these format differences in your analysis. 
        Compare the resumes against the provided job description to assess their relevance and fit.""",
        agent=resume_review_agent,
        output_file='output/jd.txt',
        expected_output="A detailed analysis of both resumes, highlighting strengths and weaknesses in relation to the job description."
    )

        # crew = Crew(
        #     agents=[
        #         agents.resume_strategist(),
        #         agents.cover_letter_strategist(),
        #         agents.document_validation_manager()
        #     ],
        #     tasks=[
        #         tasks.resume_strategy_task(agents.resume_strategist()),
        #         tasks.cover_letter_strategy_task(agents.cover_letter_strategist()),
        #         tasks.document_validation_task(agents.document_validation_manager())
        #     ],
        #     process = Process.hierarchical, # Hierarchical process to manage delegation
        #     manager_llm = manager_llm_35_turbo,  # Assign the manager_llm to the Crew
        #     verbose=True
        # )
    resume_review_crew = Crew(
        agents=[resume_review_agent],
        tasks=[review_task, compare_task, suggest_task, rate_task, rewrite_task, reflect_task],
        verbose=2,
        process=Process.sequential,
        manager_llm = manager_llm_35_turbo
    )

    return resume_review_crew


def run_resume_review(original_resume_path, modified_resume_path, jd_url, output_path):
    original_resume = read_markdown_file(original_resume_path)
    modified_resume = read_pdf_file(modified_resume_path)
    # job_description = fetch_job_description(jd_url)
    job_description = read_text_file(jd_url)
    crew = create_resume_review_agent()
    result = crew.kickoff(
        inputs={
            "original_resume": original_resume,
            "modified_resume": modified_resume,
            "job_description": job_description
        }
    )

    # Extract the rewritten resume from the result
    rewritten_resume = None
    for task_result in result:
        if "rewrite_task" in task_result:
            rewritten_resume = task_result["rewrite_task"]
            break

    if rewritten_resume:
        # Convert the Markdown content to PDF
        markdown_to_pdf(rewritten_resume, output_path)
        print(f"Rewritten resume saved as PDF to {output_path}")
    else:
        print("No rewritten resume found in the results.")
    return result


if __name__ == "__main__":
    original_resume_path = "output/updated_resume.md"
    modified_resume_path = "output/updated_resume.pdf"
    # job_description_url = "https://www.amazon.jobs/en/jobs/2558200/principal-applied-scientist"
    # job_description_url = 'https://www.google.com/about/careers/applications/jobs/results/129357347087622854-staff-machine-learning-software-engineer-ai-innovationresearch'
    job_description_url = 'output/jd.txt'
    output_resume_path = "output/rewritten_resume.pdf"
    
    review_result = run_resume_review(original_resume_path, modified_resume_path, job_description_url, output_resume_path)
    print(review_result)